# 01 — Data Preprocessing & Imputation

**Pipeline position:** Step 1 of 7

**Purpose:** Impute missing values across 137 variables (N ≈ 402,226) using
optimised Random Forest imputation.  Two scripts are provided:
- **Cell A** — single-pass imputation on the already-split dataset (recommended)
- **Cell B** — full pipeline: split then impute each split separately (data-leakage-free)

**License:** MIT

## Prerequisites
- Raw data file: `./Data/dataset1008.csv`

## Outputs
- `./Data/data_training_imputed.csv`
- `./Data/data_internal_validation_imputed.csv`
- `./Data/data_temporal_validation_imputed.csv`
- `./Data/imputed_data.csv` (combined)


In [ ]:
# SPDX-License-Identifier: MIT
# ============================================================
# Dependency check — run this cell first
# ============================================================
import importlib.util

_required = ['pandas', 'numpy', 'sklearn', 'openpyxl', 'joblib']
_missing = [p for p in _required if importlib.util.find_spec(p) is None]
if _missing:
    raise ImportError(
        f"Missing packages: {_missing}\n"
        f"Install with: pip install {' '.join(_missing)}"
    )
print("Core dependencies satisfied.")

In [ ]:
# ============================================================
# Global constants — modify paths to match your environment
# ============================================================
RANDOM_STATE     = 42          # reproducibility seed (standardised across pipeline)
OUTCOME_VAR      = "SpontAbortion"
INDEX_VAR        = "Index"
DATE_VARS        = ["BaseInfoDate"]
RAW_DATA_PATH    = "./Data/dataset1008.csv"
OUTPUT_DIR       = "./Data"


In [1]:
"""
Large-sample mixed-type missing value imputation (performance-optimised)
=======================================================================
Dataset: Pre-pregnancy examination — Spontaneous Abortion Prediction
(N ≈ 402,226, P = 137)

Optimisation strategy:
  1. Block imputation: group variables by missingness rate, avoid single pass
  2. Subsampled training: fit RF on random subset to reduce compute cost
  3. Lightweight forest: fewer trees/depth, balancing accuracy vs speed
  4. Parallel computation: leverage all available CPU cores

Method:
  - Continuous variables → RandomForestRegressor
  - Categorical (ordinal/nominal/binary) → RandomForestClassifier

Dependencies:
  pip install pandas numpy scikit-learn openpyxl joblib
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import warnings
import time
import gc

warnings.filterwarnings("ignore")
np.random.seed(42)

# ============================================================
# 1. Variable type definitions
# ============================================================
OUTCOME_VAR = "SpontAbortion"
INDEX_VAR = "Index"
DATE_VARS = ["BaseInfoDate"]

CONTINUOUS_VARS = [
    "WifeAge", "HusbandAge", "MenarcheAge",
    "WifeHeight", "WifeBMI", "WifeHR", "WifeSBP", "WifeDBP",
    "HusbHeight", "HusbBMI", "HusbHR", "HusbSBP", "HusbDBP",
    "Hemoglobin", "RBC", "Platelet", "WBC",
    "NeutrophilPct", "LymphocytePct",
    "Glucose", "WifeALT", "WifeCreat", "TSH",
    "HusbALT", "HusbCreat",
    "LeftTestVol", "RightTestVol",
]

ORDINAL_VARS = ["WifeEdu", "HusbandEdu", "WifeOcc", "HusbandOcc"]

NOMINAL_VARS = [
    "WifeEthnic", "HusbandEthnic",
    "WifeResident", "HusbandResident",
    "WifeABO", "HusbABO",
    "CycleRegular", "MenstrualVol", "Dysmenorrhea",
    "WifeMental", "WifeIntel", "WifeFace", "WifePosture",
    "WifeFaceSpec", "WifeSkinHair",
    "HusbMental", "HusbIntel", "HusbFace", "HusbPosture",
    "HusbFaceSpec", "HusbSkinHair",
    "WifePubHair", "Breast", "Vulva",
    "HusbPubHair", "AdamsApple", "Penis", "Foreskin",
    "Testis", "Epididymis", "VasDeferens", "Varicocele",
    "WifeUrine", "HusbUrine",
    "WifeHepB", "HusbHepB",
]

BINARY_VARS = [
    "WifeDiseaseHx", "WifeBirthDefect", "GynDisease",
    "WifeMedication", "WifeVaccine", "Contraception",
    "PrevPregnancy", "Consanguinity", "WifeFamConsang", "WifeFamDisease",
    "WifeMeatEgg", "WifeVegAverse", "WifeRawMeat",
    "WifeSmoke", "WifePassSmoke", "WifeAlcohol",
    "WifeDrugUse", "WifeHalitosis", "WifeGumBleed",
    "WifeEnvExpose", "WifeStress", "WifeRelatStress",
    "WifeFinance", "WifeReady",
    "WifeThyroid", "WifeLung", "WifeRhythm", "WifeMurmur",
    "WifeLiverSpleen", "WifeExtremity", "WifeGynExam",
    "WifeRh", "HusbRh",
    "RubellaIgG", "CMVIgG", "CMVIgM",
    "ToxoIgG", "ToxoIgM",
    "SyphilisScr", "HusbSyphilis",
    "GynUltrasound",
    "HusbDiseaseHis", "HusbBirthDefect", "HusbAndrologyDis",
    "HusbMedication", "HusbVaccinated",
    "HusbFamConsang", "HusbFamDisease",
    "HusbMeatEgg", "HusbVegAverse", "HusbRawMeat",
    "HusbSmoke", "HusbPassSmoke", "HusbAlcohol", "HusbDrugUse",
    "HusbEnvExpose", "HusbStress", "HusbRelatStress",
    "HusbFinance", "HusbReady",
    "HusbThyroid", "HusbLung", "HusbRhythm", "HusbMurmur",
    "HusbLiverSpleen", "HusbExtremity", "HusbAndroExam",
]

# ============================================================
# 2. Performance parameters (adjust to your hardware)
# ============================================================
class ImputationConfig:
    """Imputation hyperparameter configuration"""
    # Random Forest parameters
    N_ESTIMATORS = 50          # Number of trees (50 is sufficient for large samples)
    MAX_DEPTH = 15             # Max depth to prevent overfitting and speed up training
    MAX_FEATURES = "sqrt"      # Number of features per tree
    MIN_SAMPLES_LEAF = 20      # Minimum samples per leaf

    # Sub-sampling strategy
    SUBSAMPLE_SIZE = 500000     # Max training subsample size (100K drawn from 400K)
    PRED_BATCH_SIZE = 500000    # Batch size for prediction (avoids memory overflow)

    # Iteration control
    MAX_ITER = 5               # Max MICE iterations (3-5 rounds usually sufficient for large datasets)
    CONV_THRESHOLD_CONT = 1e-4 # Convergence threshold for continuous variables
    CONV_THRESHOLD_CAT = 1e-3  # Convergence threshold for categorical variables

    # Missing value exclusion threshold
    MISSING_THRESHOLD = 0.40   # Drop features with >40% missing values

    # Parallelism
    N_JOBS = -1                # Use all available CPU cores

    RANDOM_STATE = 42


CONFIG = ImputationConfig()

# ============================================================
# 3. Load data
# ============================================================
FILE_PATH = "./Data/dataset1008.csv"  # <-- Update to your actual file path

print("Loading data...")
t0 = time.time()
if FILE_PATH.endswith(".csv"):
    df_raw = pd.read_csv(FILE_PATH)
else:
    df_raw = pd.read_excel(FILE_PATH)
print(f"Data loaded in {time.time()-t0:.1f}s")
print(f"Raw data shape: {df_raw.shape}")
print(f"Total missing values: {df_raw.isnull().sum().sum()}")

# ============================================================
# 4. Data preprocessing
# ============================================================
exclude_vars = [INDEX_VAR] + DATE_VARS
df = df_raw.drop(columns=[c for c in exclude_vars if c in df_raw.columns], errors="ignore")

# Exclude rows with missing outcome
n_before = len(df)
if OUTCOME_VAR in df.columns:
    df = df.dropna(subset=[OUTCOME_VAR])
print(f"\nExcluding rows with missing outcome: {n_before} → {len(df)}")

# Encode binary variables
for col in BINARY_VARS + [OUTCOME_VAR]:
    if col in df.columns:
        df[col] = df[col].map({
            True: 1, False: 0, "TRUE": 1, "FALSE": 0,
            1: 1, 0: 0, 1.0: 1, 0.0: 0
        })

# Identify columns present in the loaded dataframe
actual_continuous = [c for c in CONTINUOUS_VARS if c in df.columns]
actual_ordinal = [c for c in ORDINAL_VARS if c in df.columns]
actual_nominal = [c for c in NOMINAL_VARS if c in df.columns]
actual_binary = [c for c in BINARY_VARS if c in df.columns]
all_categorical = actual_ordinal + actual_nominal + actual_binary

# ============================================================
# 5. Missing value report
# ============================================================
print(f"\n{'='*70}")
print("Missing value report")
print(f"{'='*70}")

missing_info = []
for col in df.columns:
    n_miss = df[col].isnull().sum()
    if n_miss > 0:
        pct = n_miss / len(df) * 100
        if col in actual_continuous:
            vtype = "Continuous"
        elif col in actual_ordinal:
            vtype = "Ordinal"
        elif col in actual_nominal:
            vtype = "Nominal"
        elif col in actual_binary:
            vtype = "Binary"
        else:
            vtype = "Other"
        missing_info.append({
            "Variable": col, "Type": vtype,
            "Missing Count": n_miss, "Missing Rate (%)": round(pct, 2)
        })

if missing_info:
    missing_report = pd.DataFrame(missing_info).sort_values("Missing Rate (%)", ascending=False)
    print(missing_report.to_string(index=False))

# Drop features exceeding missing-value threshold
cols_high_missing = [
    col for col in df.columns
    if df[col].isnull().mean() > CONFIG.MISSING_THRESHOLD and col != OUTCOME_VAR
]
if cols_high_missing:
    print(f"\nDropping features with >{CONFIG.MISSING_THRESHOLD*100:.0f}% missing: {cols_high_missing}")
    df = df.drop(columns=cols_high_missing)
    actual_continuous = [c for c in actual_continuous if c not in cols_high_missing]
    actual_ordinal = [c for c in actual_ordinal if c not in cols_high_missing]
    actual_nominal = [c for c in actual_nominal if c not in cols_high_missing]
    actual_binary = [c for c in actual_binary if c not in cols_high_missing]
    all_categorical = actual_ordinal + actual_nominal + actual_binary
else:
    print("\nAll features are within the missing-value threshold; none dropped.")

# ============================================================
# 6. Encode categorical variables
# ============================================================
print("\nEncoding categorical variables...")
label_encoders = {}
df_encoded = df.copy()

for col in actual_ordinal + actual_nominal:
    le = LabelEncoder()
    non_null = df_encoded[col].notnull()
    if non_null.sum() > 0:
        le.fit(df_encoded.loc[non_null, col].astype(str))
        df_encoded.loc[non_null, col] = le.transform(
            df_encoded.loc[non_null, col].astype(str)
        )
        label_encoders[col] = le

df_encoded = df_encoded.apply(pd.to_numeric, errors="coerce")

# ============================================================
# 7. Core: large-sample optimised RF iterative imputation
# ============================================================
def subsample_indices(mask_obs, n_sample, rng):
    """Sample indices from non-missing observations for sub-sampled RF training."""
    obs_idx = np.where(mask_obs)[0]
    if len(obs_idx) <= n_sample:
        return obs_idx
    return rng.choice(obs_idx, size=n_sample, replace=False)


def predict_in_batches(model, X, batch_size):
    """Run model predictions in batches to avoid memory overflow."""
    n = len(X)
    if n <= batch_size:
        return model.predict(X)
    predictions = []
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        predictions.append(model.predict(X[start:end]))
    return np.concatenate(predictions)


def optimized_rf_imputation(df_enc, continuous_cols, categorical_cols, config):
    """
    Large-sample optimised Random Forest iterative imputation.

    Optimisation strategy:
    1. Sub-sampled training: fit RF on a random subset (100K from 400K)
    2. Limit tree depth and leaf size: MAX_DEPTH=15, MIN_SAMPLES_LEAF=20
    3. Fewer trees: N_ESTIMATORS=50
    4. Batch prediction: avoids memory overflow for large matrices
    5. Sort variables by missingness: impute least-missing first
    """
    df_imp = df_enc.copy()
    all_cols = list(df_imp.columns)
    rng = np.random.RandomState(config.RANDOM_STATE)

    # Convert to numpy for speed (keep column name mapping)
    col_to_idx = {col: i for i, col in enumerate(all_cols)}
    data = df_imp.values.astype(np.float64)

    # Record missing value positions
    missing_mask = np.isnan(data)
    cols_with_missing = [c for c in all_cols if missing_mask[:, col_to_idx[c]].any()]

    if not cols_with_missing:
        print("No missing values detected.")
        return df_imp

    # Initial fill (median / mode)
    print("\nInitial fill with median / mode...")
    for col in cols_with_missing:
        idx = col_to_idx[col]
        col_data = data[:, idx]
        obs_vals = col_data[~np.isnan(col_data)]
        if col in continuous_cols:
            fill_val = np.median(obs_vals) if len(obs_vals) > 0 else 0
        else:
            if len(obs_vals) > 0:
                vals, counts = np.unique(obs_vals, return_counts=True)
                fill_val = vals[np.argmax(counts)]
            else:
                fill_val = 0
        data[missing_mask[:, idx], idx] = fill_val

    # Sort variables by number of missing values
    cols_with_missing.sort(key=lambda c: missing_mask[:, col_to_idx[c]].sum())

    n_cont = len([c for c in cols_with_missing if c in continuous_cols])
    n_cat = len([c for c in cols_with_missing if c in categorical_cols])

    print(f"\n{'='*70}")
    print(f"Starting large-sample optimised RF iterative imputation")
    print(f"{'='*70}")
    print(f"  Samples: {len(data):,}")
    print(f"  Variables to impute: {len(cols_with_missing)} "
          f"(continuous {n_cont}, categorical {n_cat})")
    print(f"  Training sub-sample size: {config.SUBSAMPLE_SIZE:,}")
    print(f"  Random Forest: {config.N_ESTIMATORS} trees, "
          f"max depth {config.MAX_DEPTH}")
    print(f"  Max iterations: {config.MAX_ITER}")
    print(f"{'='*70}")

    all_feature_indices = np.arange(len(all_cols))

    for iteration in range(config.MAX_ITER):
        iter_start = time.time()
        data_prev = data.copy()
        var_count = 0

        for col in cols_with_missing:
            var_count += 1
            col_idx = col_to_idx[col]
            col_missing = missing_mask[:, col_idx]
            n_missing = col_missing.sum()

            if n_missing == 0:
                continue

            # Feature column indices (all except current target column)
            feat_idx = all_feature_indices[all_feature_indices != col_idx]

            # --- Sub-sampled RF training ---
            obs_mask = ~col_missing
            train_indices = subsample_indices(obs_mask, config.SUBSAMPLE_SIZE, rng)

            X_train = data[train_indices][:, feat_idx]
            y_train = data[train_indices, col_idx]

            # Prediction set
            pred_indices = np.where(col_missing)[0]
            X_pred = data[pred_indices][:, feat_idx]

            if col in continuous_cols:
                model = RandomForestRegressor(
                    n_estimators=config.N_ESTIMATORS,
                    max_depth=config.MAX_DEPTH,
                    max_features=config.MAX_FEATURES,
                    min_samples_leaf=config.MIN_SAMPLES_LEAF,
                    random_state=config.RANDOM_STATE,
                    n_jobs=config.N_JOBS,
                )
                model.fit(X_train, y_train)
                predicted = predict_in_batches(model, X_pred, config.PRED_BATCH_SIZE)
            else:
                model = RandomForestClassifier(
                    n_estimators=config.N_ESTIMATORS,
                    max_depth=config.MAX_DEPTH,
                    max_features=config.MAX_FEATURES,
                    min_samples_leaf=config.MIN_SAMPLES_LEAF,
                    random_state=config.RANDOM_STATE,
                    n_jobs=config.N_JOBS,
                )
                model.fit(X_train, y_train.astype(int))
                predicted = predict_in_batches(
                    model, X_pred, config.PRED_BATCH_SIZE
                ).astype(float)

            data[pred_indices, col_idx] = predicted

            # Free memory
            del model, X_train, y_train, X_pred, predicted
            if var_count % 20 == 0:
                gc.collect()
                elapsed = time.time() - iter_start
                print(f"    ... Iter {iteration+1}: "
                      f"{var_count}/{len(cols_with_missing)} variables processed, "
                      f"{elapsed:.0f}s elapsed")

        # --- Convergence check ---
        cont_col_indices = [col_to_idx[c] for c in cols_with_missing
                            if c in continuous_cols]
        cat_col_indices = [col_to_idx[c] for c in cols_with_missing
                           if c in categorical_cols]

        if cont_col_indices:
            old_c = data_prev[:, cont_col_indices]
            new_c = data[:, cont_col_indices]
            denom = np.sum(new_c ** 2) + 1e-10
            cont_diff = np.sum((new_c - old_c) ** 2) / denom
        else:
            cont_diff = 0.0

        if cat_col_indices:
            old_cat = data_prev[:, cat_col_indices]
            new_cat = data[:, cat_col_indices]
            cat_diff = np.mean(old_cat != new_cat)
        else:
            cat_diff = 0.0

        iter_time = time.time() - iter_start
        print(f"\n  Iter {iteration+1} complete | "
              f"Continuous delta: {cont_diff:.6f} | "
              f"Categorical mismatch: {cat_diff:.6f} | "
              f"Elapsed: {iter_time:.0f}s")

        del data_prev
        gc.collect()

        if (cont_diff < config.CONV_THRESHOLD_CONT and
                cat_diff < config.CONV_THRESHOLD_CAT):
            print(f"\n  ✓ Converged at iteration {iteration+1}!")
            break

    # Write imputed values back to DataFrame
    df_imp = pd.DataFrame(data, columns=all_cols, index=df_enc.index)
    return df_imp


# Run imputation
total_start = time.time()
df_imputed = optimized_rf_imputation(
    df_encoded,
    continuous_cols=actual_continuous,
    categorical_cols=all_categorical,
    config=CONFIG
)
total_time = time.time() - total_start
print(f"\nTotal imputation time: {total_time/60:.1f} min")

# ============================================================
# 8. Decode labels back to original values
# ============================================================
print("\nDecoding labels back to original values...")
df_result = df_imputed.copy()

# Restore binary variable labels
for col in actual_binary:
    if col in df_result.columns:
        df_result[col] = df_result[col].round().astype(int).map({1: True, 0: False})

# Restore outcome variable
if OUTCOME_VAR in df_result.columns:
    df_result[OUTCOME_VAR] = (
        df_result[OUTCOME_VAR].round().astype(int).map({1: True, 0: False})
    )

# Restore ordinal / nominal categories
for col in actual_ordinal + actual_nominal:
    if col in label_encoders and col in df_result.columns:
        le = label_encoders[col]
        max_label = len(le.classes_) - 1
        df_result[col] = df_result[col].round().clip(0, max_label).astype(int)
        df_result[col] = le.inverse_transform(df_result[col])

# Round continuous integer variables
INTEGER_CONTINUOUS = [
    "WifeAge", "HusbandAge", "MenarcheAge",
    "WifeHR", "WifeSBP", "WifeDBP",
    "HusbHR", "HusbSBP", "HusbDBP",
    "WifeHeight", "HusbHeight",
    "LeftTestVol", "RightTestVol",
]
for col in INTEGER_CONTINUOUS:
    if col in df_result.columns:
        df_result[col] = df_result[col].round().astype(int)

# Restore Index and Date columns
if INDEX_VAR in df_raw.columns:
    df_result.insert(0, INDEX_VAR, df_raw.loc[df_result.index, INDEX_VAR].values)
for dcol in DATE_VARS:
    if dcol in df_raw.columns:
        df_result[dcol] = df_raw.loc[df_result.index, dcol].values

# ============================================================
# 9. Save results
# ============================================================
OUTPUT_PATH = "./Data/imputed_data.csv"  # CSV recommended for large datasets (avoids Excel row limit)
df_result.to_csv(OUTPUT_PATH, index=False)

# ============================================================
# 10. Summary report
# ============================================================
print(f"\n{'='*70}")
print("Imputation complete — summary")
print(f"{'='*70}")
print(f"  Samples: {len(df_result):,}")
print(f"  Variables: {df_result.shape[1]}")
print(f"  Missing before imputation: {df_raw.drop(columns=[INDEX_VAR]+DATE_VARS, errors='ignore').isnull().sum().sum():,}")
print(f"  Missing after imputation:  {df_result.drop(columns=[INDEX_VAR]+DATE_VARS, errors='ignore').isnull().sum().sum():,}")
print(f"  Total elapsed: {total_time/60:.1f} min")
print(f"\n  Variable types and imputation methods:")
print(f"    Continuous  : {len(actual_continuous):3d} → RandomForestRegressor")
print(f"    Ordinal     : {len(actual_ordinal):3d} → RandomForestClassifier")
print(f"    Nominal     : {len(actual_nominal):3d} → RandomForestClassifier")
print(f"    Binary      : {len(actual_binary):3d} → RandomForestClassifier")
print(f"\n  Performance configuration:")
print(f"    Training sub-sample: {CONFIG.SUBSAMPLE_SIZE:,} / {len(df_result):,}")
print(f"    Forest: {CONFIG.N_ESTIMATORS} trees, max_depth {CONFIG.MAX_DEPTH}")
print(f"    Max iterations: {CONFIG.MAX_ITER}")
print(f"\n  Output file: {OUTPUT_PATH}")
print(f"{'='*70}")

正在读取数据...
读取完成, 耗时 3.0s
原始数据维度: (402226, 137)
总缺失值数量: 2260644

排除结局缺失: 402226 → 402226 例

缺失值报告
              变量   类型   缺失数  缺失比例(%)
      HusbandOcc 有序分类 79299    19.72
         WifeOcc 有序分类 76614    19.05
           Vulva 无序分类 74237    18.46
     WifePubHair 无序分类 65822    16.36
          Breast 无序分类 65712    16.34
   GynUltrasound  二分类 62223    15.47
    RightTestVol   连续 55760    13.86
     LeftTestVol   连续 55745    13.86
          Testis 无序分类 47396    11.78
           Penis 无序分类 43884    10.91
        Foreskin 无序分类 43666    10.86
      Varicocele 无序分类 43325    10.77
     VasDeferens 无序分类 43234    10.75
      HusbandEdu 有序分类 43201    10.74
      Epididymis 无序分类 43013    10.69
     HusbPubHair 无序分类 42624    10.60
         WifeEdu 有序分类 42509    10.57
  HusbDiseaseHis  二分类 42441    10.55
      AdamsApple 无序分类 40830    10.15
       WifeUrine 无序分类 38165     9.49
       HusbUrine 无序分类 36882     9.17
        HusbHepB 无序分类 34785     8.65
   NeutrophilPct   连续 26193     6.51
   LymphocytePct

In [2]:
"""
Complete pipeline: split data first → then impute each split separately
======================================================================
Correct order:
  Step 1: Split raw data by year and stratified sampling into three splits
  Step 2: Fit imputation models independently on the training set
  Step 3: Initialise validation sets using training-set median/mode
  Step 4: Run RF iterative imputation separately for each split
  Step 5: Boruta feature selection (training set only)

This approach prevents data leakage and ensures unbiased validation.

Dependencies:
  pip install pandas numpy scikit-learn boruta openpyxl
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import warnings
import time
import gc

warnings.filterwarnings("ignore")

# ============================================================
# Global configuration
# ============================================================
OUTCOME_VAR = "SpontAbortion"
INDEX_VAR = "Index"
DATE_VAR = "BaseInfoDate"
RANDOM_STATE = 42
TRAIN_RATIO = 0.9

# Variable type definitions
CONTINUOUS_VARS = [
    "WifeAge", "HusbandAge", "MenarcheAge",
    "WifeHeight", "WifeBMI", "WifeHR", "WifeSBP", "WifeDBP",
    "HusbHeight", "HusbBMI", "HusbHR", "HusbSBP", "HusbDBP",
    "Hemoglobin", "RBC", "Platelet", "WBC",
    "NeutrophilPct", "LymphocytePct",
    "Glucose", "WifeALT", "WifeCreat", "TSH",
    "HusbALT", "HusbCreat",
    "LeftTestVol", "RightTestVol",
]

ORDINAL_VARS = ["WifeEdu", "HusbandEdu", "WifeOcc", "HusbandOcc"]

NOMINAL_VARS = [
    "WifeEthnic", "HusbandEthnic",
    "WifeResident", "HusbandResident",
    "WifeABO", "HusbABO",
    "CycleRegular", "MenstrualVol", "Dysmenorrhea",
    "WifeMental", "WifeIntel", "WifeFace", "WifePosture",
    "WifeFaceSpec", "WifeSkinHair",
    "HusbMental", "HusbIntel", "HusbFace", "HusbPosture",
    "HusbFaceSpec", "HusbSkinHair",
    "WifePubHair", "Breast", "Vulva",
    "HusbPubHair", "AdamsApple", "Penis", "Foreskin",
    "Testis", "Epididymis", "VasDeferens", "Varicocele",
    "WifeUrine", "HusbUrine",
    "WifeHepB", "HusbHepB",
]

BINARY_VARS = [
    "WifeDiseaseHx", "WifeBirthDefect", "GynDisease",
    "WifeMedication", "WifeVaccine", "Contraception",
    "PrevPregnancy", "Consanguinity", "WifeFamConsang", "WifeFamDisease",
    "WifeMeatEgg", "WifeVegAverse", "WifeRawMeat",
    "WifeSmoke", "WifePassSmoke", "WifeAlcohol",
    "WifeDrugUse", "WifeHalitosis", "WifeGumBleed",
    "WifeEnvExpose", "WifeStress", "WifeRelatStress",
    "WifeFinance", "WifeReady",
    "WifeThyroid", "WifeLung", "WifeRhythm", "WifeMurmur",
    "WifeLiverSpleen", "WifeExtremity", "WifeGynExam",
    "WifeRh", "HusbRh",
    "RubellaIgG", "CMVIgG", "CMVIgM",
    "ToxoIgG", "ToxoIgM",
    "SyphilisScr", "HusbSyphilis",
    "GynUltrasound",
    "HusbDiseaseHis", "HusbBirthDefect", "HusbAndrologyDis",
    "HusbMedication", "HusbVaccinated",
    "HusbFamConsang", "HusbFamDisease",
    "HusbMeatEgg", "HusbVegAverse", "HusbRawMeat",
    "HusbSmoke", "HusbPassSmoke", "HusbAlcohol", "HusbDrugUse",
    "HusbEnvExpose", "HusbStress", "HusbRelatStress",
    "HusbFinance", "HusbReady",
    "HusbThyroid", "HusbLung", "HusbRhythm", "HusbMurmur",
    "HusbLiverSpleen", "HusbExtremity", "HusbAndroExam",
]

# Imputation parameters
class ImputationConfig:
    N_ESTIMATORS = 50
    MAX_DEPTH = 15
    MAX_FEATURES = "sqrt"
    MIN_SAMPLES_LEAF = 20
    SUBSAMPLE_SIZE = 500000
    PRED_BATCH_SIZE = 500000
    MAX_ITER = 5
    CONV_THRESHOLD_CONT = 1e-4
    CONV_THRESHOLD_CAT = 1e-3
    MISSING_THRESHOLD = 0.40
    N_JOBS = -1
    RANDOM_STATE = 42

CONFIG = ImputationConfig()


# ============================================================
# STEP 1: Load raw data and split into dataset splits
# ============================================================
print("=" * 70)
print("STEP 1: Load raw data and split into dataset splits")
print("=" * 70)

FILE_PATH = "./Data/dataset1008.csv"  # <-- Update to your actual raw data file path

if FILE_PATH.endswith(".csv"):
    df_raw = pd.read_csv(FILE_PATH)
else:
    df_raw = pd.read_excel(FILE_PATH)
print(f"Raw data shape: {df_raw.shape}")

# Exclude rows with missing outcome
df_raw = df_raw.dropna(subset=[OUTCOME_VAR])
print(f"After dropping missing outcomes: {len(df_raw):,}")

# Parse year from date
df_raw[DATE_VAR] = pd.to_datetime(df_raw[DATE_VAR], errors="coerce")
df_raw["Year"] = df_raw[DATE_VAR].dt.year

# Encode outcome variable
df_raw[OUTCOME_VAR] = df_raw[OUTCOME_VAR].map({
    True: 1, False: 0, "TRUE": 1, "FALSE": 0, 1: 1, 0: 0
})

# 1.1 Temporal validation set (year 2019)
df_temporal = df_raw[df_raw["Year"] == 2019].copy()
df_dev = df_raw[df_raw["Year"].between(2014, 2018) | df_raw["Year"].isnull()].copy()

# 1.2 Stratified 9:1 split of development set
df_train, df_internal_val = train_test_split(
    df_dev,
    test_size=1 - TRAIN_RATIO,
    stratify=df_dev[OUTCOME_VAR].astype(int),
    random_state=RANDOM_STATE,
)

print(f"\nDataset split summary:")
for name, data in [("Training", df_train), ("Internal Validation", df_internal_val),
                    ("Temporal Validation", df_temporal)]:
    y = data[OUTCOME_VAR].astype(int)
    print(f"  {name:12s}: N={len(data):>7,}  |  "
          f"SA={y.sum():>5,} ({y.mean()*100:.2f}%)")

# Drop auxiliary columns
for subset in [df_train, df_internal_val, df_temporal]:
    subset.drop(columns=["Year"], inplace=True, errors="ignore")


# ============================================================
# STEP 2: Encoding functions (shared encoder across all splits)
# ============================================================
print(f"\n{'='*70}")
print("STEP 2: Encode categorical variables (fit on training set only)")
print("=" * 70)

def encode_dataset(df, label_encoders=None, fit=False):
    """
    Encode a dataset.
    fit=True  — fit LabelEncoder on training set
    fit=False — apply pre-fitted encoder to validation sets
    """
    df_enc = df.copy()
    if label_encoders is None:
        label_encoders = {}

    # Encode binary variables
    for col in BINARY_VARS:
        if col in df_enc.columns:
            df_enc[col] = df_enc[col].map({
                True: 1, False: 0, "TRUE": 1, "FALSE": 0, 1: 1, 0: 0
            })

    # Encode categorical columns
    actual_cat = [c for c in ORDINAL_VARS + NOMINAL_VARS if c in df_enc.columns]
    for col in actual_cat:
        if fit:
            le = LabelEncoder()
            non_null = df_enc[col].notnull()
            if non_null.sum() > 0:
                # Fit on all observed categories in the training set
                le.fit(df_enc.loc[non_null, col].astype(str))
                df_enc.loc[non_null, col] = le.transform(
                    df_enc.loc[non_null, col].astype(str)
                )
                label_encoders[col] = le
        else:
            if col in label_encoders:
                le = label_encoders[col]
                non_null = df_enc[col].notnull()
                if non_null.sum() > 0:
                    # Map unknown categories (not seen in training) to NaN
                    known_classes = set(le.classes_)
                    df_enc.loc[non_null, col] = df_enc.loc[non_null, col].astype(str).apply(
                        lambda x: le.transform([x])[0] if x in known_classes else np.nan
                    )

    # Drop non-feature columns
    drop_cols = [INDEX_VAR, DATE_VAR]
    df_enc = df_enc.drop(columns=[c for c in drop_cols if c in df_enc.columns],
                          errors="ignore")
    df_enc = df_enc.apply(pd.to_numeric, errors="coerce")

    return df_enc, label_encoders


# Fit encoders on training set
df_train_enc, label_encoders = encode_dataset(df_train, fit=True)
# Apply training-set encoders to validation and test sets
df_intval_enc, _ = encode_dataset(df_internal_val, label_encoders=label_encoders, fit=False)
df_tempval_enc, _ = encode_dataset(df_temporal, label_encoders=label_encoders, fit=False)

print("Encoding complete (encoder fitted on training set, applied to validation sets)")


# ============================================================
# STEP 3: Imputation functions
# ============================================================
def get_actual_cols(df):
    """Return variable lists filtered to columns present in df."""
    cont = [c for c in CONTINUOUS_VARS if c in df.columns]
    ordi = [c for c in ORDINAL_VARS if c in df.columns]
    nomi = [c for c in NOMINAL_VARS if c in df.columns]
    bina = [c for c in BINARY_VARS if c in df.columns]
    return cont, ordi + nomi + bina


def subsample_indices(mask_obs, n_sample, rng):
    obs_idx = np.where(mask_obs)[0]
    if len(obs_idx) <= n_sample:
        return obs_idx
    return rng.choice(obs_idx, size=n_sample, replace=False)


def predict_in_batches(model, X, batch_size=50000):
    n = len(X)
    if n <= batch_size:
        return model.predict(X)
    preds = []
    for s in range(0, n, batch_size):
        preds.append(model.predict(X[s:min(s + batch_size, n)]))
    return np.concatenate(preds)


def rf_impute(df_enc, dataset_name, config=CONFIG):
    """
    Run Random Forest iterative imputation for a single dataset split.
    """
    continuous_cols, categorical_cols = get_actual_cols(df_enc)
    all_cols = list(df_enc.columns)
    col_to_idx = {col: i for i, col in enumerate(all_cols)}

    data = df_enc.values.astype(np.float64)
    missing_mask = np.isnan(data)

    total_missing = missing_mask.sum()
    if total_missing == 0:
        print(f"  [{dataset_name}] No missing values — skipping imputation.")
        return df_enc

    cols_with_missing = [c for c in all_cols if missing_mask[:, col_to_idx[c]].any()]
    cols_with_missing.sort(key=lambda c: missing_mask[:, col_to_idx[c]].sum())

    rng = np.random.RandomState(config.RANDOM_STATE)

    # Initial fill (median / mode)
    for col in cols_with_missing:
        idx = col_to_idx[col]
        obs_vals = data[~np.isnan(data[:, idx]), idx]
        if col in continuous_cols:
            fill = np.median(obs_vals) if len(obs_vals) > 0 else 0
        else:
            if len(obs_vals) > 0:
                vals, counts = np.unique(obs_vals, return_counts=True)
                fill = vals[np.argmax(counts)]
            else:
                fill = 0
        data[missing_mask[:, idx], idx] = fill

    n_cont = len([c for c in cols_with_missing if c in continuous_cols])
    n_cat = len([c for c in cols_with_missing if c in categorical_cols])

    print(f"\n  [{dataset_name}] Samples: {len(data):,} | "
          f"Missing: {total_missing:,} | "
          f"Variables: {len(cols_with_missing)} (cont {n_cont}, cat {n_cat})")

    all_feat_idx = np.arange(len(all_cols))

    for iteration in range(config.MAX_ITER):
        t0 = time.time()
        data_prev = data.copy()
        count = 0

        for col in cols_with_missing:
            count += 1
            cidx = col_to_idx[col]
            col_miss = missing_mask[:, cidx]
            if col_miss.sum() == 0:
                continue

            feat_idx = all_feat_idx[all_feat_idx != cidx]
            obs_mask = ~col_miss
            train_idx = subsample_indices(obs_mask, config.SUBSAMPLE_SIZE, rng)

            X_tr = data[train_idx][:, feat_idx]
            y_tr = data[train_idx, cidx]
            X_pr = data[np.where(col_miss)[0]][:, feat_idx]

            if col in continuous_cols:
                model = RandomForestRegressor(
                    n_estimators=config.N_ESTIMATORS, max_depth=config.MAX_DEPTH,
                    max_features=config.MAX_FEATURES, min_samples_leaf=config.MIN_SAMPLES_LEAF,
                    random_state=config.RANDOM_STATE, n_jobs=config.N_JOBS,
                )
                model.fit(X_tr, y_tr)
                pred = predict_in_batches(model, X_pr)
            else:
                model = RandomForestClassifier(
                    n_estimators=config.N_ESTIMATORS, max_depth=config.MAX_DEPTH,
                    max_features=config.MAX_FEATURES, min_samples_leaf=config.MIN_SAMPLES_LEAF,
                    random_state=config.RANDOM_STATE, n_jobs=config.N_JOBS,
                )
                model.fit(X_tr, y_tr.astype(int))
                pred = predict_in_batches(model, X_pr).astype(float)

            data[np.where(col_miss)[0], cidx] = pred
            del model, X_tr, y_tr, X_pr, pred

            if count % 20 == 0:
                gc.collect()

        # Convergence check
        cont_idx = [col_to_idx[c] for c in cols_with_missing if c in continuous_cols]
        cat_idx = [col_to_idx[c] for c in cols_with_missing if c in categorical_cols]

        cont_diff = (np.sum((data[:, cont_idx] - data_prev[:, cont_idx]) ** 2) /
                     (np.sum(data[:, cont_idx] ** 2) + 1e-10)) if cont_idx else 0
        cat_diff = np.mean(data[:, cat_idx] != data_prev[:, cat_idx]) if cat_idx else 0

        elapsed = time.time() - t0
        print(f"    Iter {iteration+1} | Cont delta: {cont_diff:.6f} | "
              f"Cat mismatch: {cat_diff:.6f} | {elapsed:.0f}s")

        del data_prev
        gc.collect()

        if cont_diff < config.CONV_THRESHOLD_CONT and cat_diff < config.CONV_THRESHOLD_CAT:
            print(f"    ✓ Converged at iteration {iteration+1}")
            break

    return pd.DataFrame(data, columns=all_cols, index=df_enc.index)


# ============================================================
# STEP 4: Run imputation separately for each dataset split
# ============================================================
print(f"\n{'='*70}")
print("STEP 3: Running RF imputation for each dataset split separately")
print("=" * 70)
print("(Each split imputed independently — no data leakage)")

total_start = time.time()

df_train_imp = rf_impute(df_train_enc, "Training")
df_intval_imp = rf_impute(df_intval_enc, "Internal Validation")
df_tempval_imp = rf_impute(df_tempval_enc, "Temporal Validation")

total_elapsed = time.time() - total_start
print(f"\nAll splits imputed. Total elapsed: {total_elapsed/60:.1f} min")


# ============================================================
# STEP 5: Decode labels back to original values
# ============================================================
print(f"\n{'='*70}")
print("STEP 4: Decoding labels back to original values")
print("=" * 70)

INTEGER_CONTINUOUS = [
    "WifeAge", "HusbandAge", "MenarcheAge",
    "WifeHR", "WifeSBP", "WifeDBP",
    "HusbHR", "HusbSBP", "HusbDBP",
    "WifeHeight", "HusbHeight",
    "LeftTestVol", "RightTestVol",
]

def decode_dataset(df_imp, label_encoders):
    """Decode: reverse LabelEncoder to restore original category strings."""
    df_out = df_imp.copy()

    # Restore binary variable labels
    for col in BINARY_VARS + [OUTCOME_VAR]:
        if col in df_out.columns:
            df_out[col] = df_out[col].round().astype(int).map({1: True, 0: False})

    # Restore categorical labels
    for col in ORDINAL_VARS + NOMINAL_VARS:
        if col in label_encoders and col in df_out.columns:
            le = label_encoders[col]
            max_label = len(le.classes_) - 1
            df_out[col] = df_out[col].round().clip(0, max_label).astype(int)
            df_out[col] = le.inverse_transform(df_out[col])

    # Round integer continuous variables
    for col in INTEGER_CONTINUOUS:
        if col in df_out.columns:
            df_out[col] = df_out[col].round().astype(int)

    return df_out


df_train_final = decode_dataset(df_train_imp, label_encoders)
df_intval_final = decode_dataset(df_intval_imp, label_encoders)
df_tempval_final = decode_dataset(df_tempval_imp, label_encoders)

# Restore Index and Date columns
for df_final, df_orig in [(df_train_final, df_train),
                           (df_intval_final, df_internal_val),
                           (df_tempval_final, df_temporal)]:
    if INDEX_VAR in df_orig.columns:
        df_final.insert(0, INDEX_VAR, df_orig.loc[df_final.index, INDEX_VAR].values)
    if DATE_VAR in df_orig.columns:
        df_final[DATE_VAR] = df_orig.loc[df_final.index, DATE_VAR].values

print("Decoding complete.")

# ============================================================
# STEP 6: Save
# ============================================================
print(f"\n{'='*70}")
print("STEP 5: Saving results")
print("=" * 70)

df_train_final["Dataset"] = "Training"
df_intval_final["Dataset"] = "Internal_Validation"
df_tempval_final["Dataset"] = "Temporal_Validation"

# Save each split separately
df_train_final.to_csv("./Data/data_training_imputed.csv", index=False)
df_intval_final.to_csv("./Data/data_internal_validation_imputed.csv", index=False)
df_tempval_final.to_csv("./Data/data_temporal_validation_imputed.csv", index=False)

# Save combined dataset
df_all = pd.concat([df_train_final, df_intval_final, df_tempval_final],
                    ignore_index=True)
df_all.to_csv("data_all_imputed_with_splits.csv", index=False)

print(f"  data_training_imputed.csv              ({len(df_train_final):,} records)")
print(f"  data_internal_validation_imputed.csv   ({len(df_intval_final):,} records)")
print(f"  data_temporal_validation_imputed.csv   ({len(df_tempval_final):,} records)")
print(f"  data_all_imputed_with_splits.csv       ({len(df_all):,} records)")

# ============================================================
# STEP 7: Summary report
# ============================================================
print(f"\n{'='*70}")
print("Summary report")
print("=" * 70)

print("\n  Correct pipeline order:")
print("  ┌─ Raw data")
print("  ├─ Step 1: Split data (by year + stratified sampling)")
print("  ├─ Step 2: Encode (encoders fitted on training set only)")
print("  ├─ Step 3: Impute each split independently (no data leakage)")
print("  ├─ Step 4: Decode back to original labels")
print("  ├─ Step 5: Save results")
print("  └─ Next: Boruta feature selection (training set only)")

print(f"\n  Dataset splits:")
for name, data in [("Training", df_train_final),
                    ("Internal Validation", df_intval_final),
                    ("Temporal Validation", df_tempval_final)]:
    y = data[OUTCOME_VAR].map({True: 1, False: 0, 1: 1, 0: 0}).astype(int)
    print(f"    {name:12s}: N={len(data):>7,} | "
          f"SA={y.sum():>5,} ({y.mean()*100:.2f}%) | "
          f"missing={data.drop(columns=[INDEX_VAR, DATE_VAR, 'Dataset'], errors='ignore').isnull().sum().sum()}")

print(f"\n  Total elapsed: {total_elapsed/60:.1f} min")
print(f"\n{'='*70}")
print("NOTE: Subsequent Boruta feature selection should run on data_training_imputed.csv only")
print("Selected features are then applied to internal/temporal validation sets for model evaluation")
print("=" * 70)

STEP 1: 读取原始数据并划分数据集
原始数据维度: (402226, 137)
排除结局缺失后: 402,226

数据集划分结果:
  训练集         : N=360,363  |  SA=11,753 (3.26%)
  内部验证集       : N= 40,041  |  SA=1,306 (3.26%)
  时间验证集       : N=  1,822  |  SA=  262 (14.38%)

STEP 2: 统一编码分类变量
编码完成（编码器在训练集上拟合，应用于验证集）

STEP 3: 分别对各数据集执行随机森林插补
（各数据集独立插补，无数据泄漏）

  [训练集] 样本: 360,363 | 总缺失: 2,024,441 | 待插补变量: 134 (连续 27, 分类 107)
    第 1 轮 | 连续变化: 0.000048 | 分类不一致: 0.002579 | 313s
    第 2 轮 | 连续变化: 0.000005 | 分类不一致: 0.000434 | 317s
    ✓ 第 2 轮收敛

  [内部验证集] 样本: 40,041 | 总缺失: 224,955 | 待插补变量: 134 (连续 27, 分类 107)
    第 1 轮 | 连续变化: 0.000039 | 分类不一致: 0.002483 | 25s
    第 2 轮 | 连续变化: 0.000006 | 分类不一致: 0.000527 | 26s
    ✓ 第 2 轮收敛

  [时间验证集] 样本: 1,822 | 总缺失: 11,248 | 待插补变量: 132 (连续 27, 分类 105)
    第 1 轮 | 连续变化: 0.000023 | 分类不一致: 0.003115 | 8s
    第 2 轮 | 连续变化: 0.000003 | 分类不一致: 0.000434 | 8s
    ✓ 第 2 轮收敛

全部插补完成，总耗时: 11.6 分钟

STEP 4: 反编码还原
反编码完成。

STEP 5: 保存结果
  data_training_imputed.csv              (360,363 例)
  data_internal_validation_imputed.csv   (40,041

In [3]:
"""
Statistical Description Script (Pre- and Post-Imputation)
===========================================================
Comprehensive descriptive statistics for three datasets (Training, Internal
Validation, Temporal Validation) before and after imputation.

Excel Report Sheets:
  1.  Dataset Overview         — Sample sizes, outcome distribution, missingness
  2.  Pre-Imputation Table 1   — Table 1 style: grouped continuous + categorical
  3.  Post-Imputation Table 1  — Same, post-imputation
  4.  Pre-Imp Continuous Vars  — Mean±SD, Median(IQR), Min-Max, KW test, SMD
  5.  Post-Imp Continuous Vars — Same
  6.  Pre-Imp Categorical Vars — n(%) per category, chi-square test, SMD
  7.  Post-Imp Categorical Vars— Same
  8.  Missing Rate Comparison  — Per-variable missingness before vs after
  9.  Imputation Effect         — Delta Mean/SD for continuous, Delta % for binary
  10. SMD Summary              — Post-imputation balance: Training vs Validation
  11. Variable Dictionary      — Variable name, label, type, group, value range

Dependencies:
  pip install pandas numpy scipy openpyxl scikit-learn
"""

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import train_test_split
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import warnings
import sys
import os

warnings.filterwarnings("ignore")

# ============================================================
# Global Configuration
# ============================================================
OUTCOME_VAR = "SpontAbortion"
INDEX_VAR = "Index"
DATE_VAR = "BaseInfoDate"
RANDOM_STATE = 42
TRAIN_RATIO = 0.9

# ============================================================
# Variable Definitions
# ============================================================
CONTINUOUS_VARS = [
    "WifeAge", "HusbandAge", "MenarcheAge",
    "WifeHeight", "WifeBMI", "WifeHR", "WifeSBP", "WifeDBP",
    "HusbHeight", "HusbBMI", "HusbHR", "HusbSBP", "HusbDBP",
    "Hemoglobin", "RBC", "Platelet", "WBC",
    "NeutrophilPct", "LymphocytePct",
    "Glucose", "WifeALT", "WifeCreat", "TSH",
    "HusbALT", "HusbCreat",
    "LeftTestVol", "RightTestVol",
]

ORDINAL_VARS = ["WifeEdu", "HusbandEdu", "WifeOcc", "HusbandOcc"]

NOMINAL_VARS = [
    "WifeEthnic", "HusbandEthnic",
    "WifeResident", "HusbandResident",
    "WifeABO", "HusbABO",
    "CycleRegular", "MenstrualVol", "Dysmenorrhea",
    "WifeMental", "WifeIntel", "WifeFace", "WifePosture",
    "WifeFaceSpec", "WifeSkinHair",
    "HusbMental", "HusbIntel", "HusbFace", "HusbPosture",
    "HusbFaceSpec", "HusbSkinHair",
    "WifePubHair", "Breast", "Vulva",
    "HusbPubHair", "AdamsApple", "Penis", "Foreskin",
    "Testis", "Epididymis", "VasDeferens", "Varicocele",
    "WifeUrine", "HusbUrine",
    "WifeHepB", "HusbHepB",
]

BINARY_VARS = [
    "WifeDiseaseHx", "WifeBirthDefect", "GynDisease",
    "WifeMedication", "WifeVaccine", "Contraception",
    "PrevPregnancy", "Consanguinity", "WifeFamConsang", "WifeFamDisease",
    "WifeMeatEgg", "WifeVegAverse", "WifeRawMeat",
    "WifeSmoke", "WifePassSmoke", "WifeAlcohol",
    "WifeDrugUse", "WifeHalitosis", "WifeGumBleed",
    "WifeEnvExpose", "WifeStress", "WifeRelatStress",
    "WifeFinance", "WifeReady",
    "WifeThyroid", "WifeLung", "WifeRhythm", "WifeMurmur",
    "WifeLiverSpleen", "WifeExtremity", "WifeGynExam",
    "WifeRh", "HusbRh",
    "RubellaIgG", "CMVIgG", "CMVIgM",
    "ToxoIgG", "ToxoIgM",
    "SyphilisScr", "HusbSyphilis",
    "GynUltrasound",
    "HusbDiseaseHis", "HusbBirthDefect", "HusbAndrologyDis",
    "HusbMedication", "HusbVaccinated",
    "HusbFamConsang", "HusbFamDisease",
    "HusbMeatEgg", "HusbVegAverse", "HusbRawMeat",
    "HusbSmoke", "HusbPassSmoke", "HusbAlcohol", "HusbDrugUse",
    "HusbEnvExpose", "HusbStress", "HusbRelatStress",
    "HusbFinance", "HusbReady",
    "HusbThyroid", "HusbLung", "HusbRhythm", "HusbMurmur",
    "HusbLiverSpleen", "HusbExtremity", "HusbAndroExam",
]

CATEGORICAL_VARS = ORDINAL_VARS + NOMINAL_VARS + BINARY_VARS

# ---- Variable Labels (English) ----
VAR_LABELS = {
    # Outcome
    "SpontAbortion": "Spontaneous Abortion",
    # Continuous
    "WifeAge": "Wife Age (years)", "HusbandAge": "Husband Age (years)",
    "MenarcheAge": "Menarche Age (years)",
    "WifeHeight": "Wife Height (cm)", "WifeBMI": "Wife BMI (kg/m2)",
    "WifeHR": "Wife Heart Rate (bpm)", "WifeSBP": "Wife SBP (mmHg)", "WifeDBP": "Wife DBP (mmHg)",
    "HusbHeight": "Husband Height (cm)", "HusbBMI": "Husband BMI (kg/m2)",
    "HusbHR": "Husband Heart Rate (bpm)", "HusbSBP": "Husband SBP (mmHg)", "HusbDBP": "Husband DBP (mmHg)",
    "Hemoglobin": "Hemoglobin (g/L)", "RBC": "Red Blood Cells (x10^12/L)",
    "Platelet": "Platelet Count (x10^9/L)", "WBC": "White Blood Cells (x10^9/L)",
    "NeutrophilPct": "Neutrophil Percentage (%)", "LymphocytePct": "Lymphocyte Percentage (%)",
    "Glucose": "Fasting Glucose (mmol/L)", "WifeALT": "Wife ALT (U/L)", "WifeCreat": "Wife Creatinine (umol/L)",
    "TSH": "Thyroid Stimulating Hormone (mIU/L)",
    "HusbALT": "Husband ALT (U/L)", "HusbCreat": "Husband Creatinine (umol/L)",
    "LeftTestVol": "Left Testicular Volume (ml)", "RightTestVol": "Right Testicular Volume (ml)",
    # Ordinal
    "WifeEdu": "Wife Education Level", "HusbandEdu": "Husband Education Level",
    "WifeOcc": "Wife Occupation", "HusbandOcc": "Husband Occupation",
    # Nominal
    "WifeEthnic": "Wife Ethnicity", "HusbandEthnic": "Husband Ethnicity",
    "WifeResident": "Wife Residency", "HusbandResident": "Husband Residency",
    "WifeABO": "Wife ABO Blood Type", "HusbABO": "Husband ABO Blood Type",
    "CycleRegular": "Menstrual Cycle Regularity", "MenstrualVol": "Menstrual Volume",
    "Dysmenorrhea": "Dysmenorrhea",
    "WifeMental": "Wife Mental Status", "WifeIntel": "Wife Intelligence",
    "WifeFace": "Wife Facial Appearance", "WifePosture": "Wife Posture",
    "WifeFaceSpec": "Wife Facial Features", "WifeSkinHair": "Wife Skin & Hair",
    "HusbMental": "Husband Mental Status", "HusbIntel": "Husband Intelligence",
    "HusbFace": "Husband Facial Appearance", "HusbPosture": "Husband Posture",
    "HusbFaceSpec": "Husband Facial Features", "HusbSkinHair": "Husband Skin & Hair",
    "WifePubHair": "Female Pubic Hair", "Breast": "Breast Development", "Vulva": "Vulva",
    "HusbPubHair": "Male Pubic Hair", "AdamsApple": "Adam's Apple", "Penis": "Penis",
    "Foreskin": "Foreskin", "Testis": "Testis", "Epididymis": "Epididymis",
    "VasDeferens": "Vas Deferens", "Varicocele": "Varicocele",
    "WifeUrine": "Wife Urinalysis", "HusbUrine": "Husband Urinalysis",
    "WifeHepB": "Wife Hepatitis B", "HusbHepB": "Husband Hepatitis B",
    # Binary
    "WifeDiseaseHx": "Wife Disease History", "WifeBirthDefect": "Wife Birth Defect History",
    "GynDisease": "Gynecological Disease", "WifeMedication": "Wife Medication History",
    "WifeVaccine": "Wife Vaccination", "Contraception": "Contraception History",
    "PrevPregnancy": "Previous Pregnancy", "Consanguinity": "Consanguineous Marriage",
    "WifeFamConsang": "Wife Family Consanguinity", "WifeFamDisease": "Wife Family Disease History",
    "WifeMeatEgg": "Wife Meat/Egg Intake", "WifeVegAverse": "Wife Vegetable Aversion",
    "WifeRawMeat": "Wife Raw Meat Consumption", "WifeSmoke": "Wife Smoking",
    "WifePassSmoke": "Wife Passive Smoking", "WifeAlcohol": "Wife Alcohol Use",
    "WifeDrugUse": "Wife Drug Use", "WifeHalitosis": "Wife Halitosis",
    "WifeGumBleed": "Wife Gum Bleeding", "WifeEnvExpose": "Wife Environmental Exposure",
    "WifeStress": "Wife Work Stress", "WifeRelatStress": "Wife Relationship Stress",
    "WifeFinance": "Wife Financial Stress", "WifeReady": "Wife Reproductive Readiness",
    "WifeThyroid": "Wife Thyroid Exam", "WifeLung": "Wife Lung Exam",
    "WifeRhythm": "Wife Cardiac Rhythm", "WifeMurmur": "Wife Heart Murmur",
    "WifeLiverSpleen": "Wife Liver/Spleen Exam", "WifeExtremity": "Wife Extremity Exam",
    "WifeGynExam": "Gynecological Exam", "WifeRh": "Wife Rh Blood Type", "HusbRh": "Husband Rh Blood Type",
    "RubellaIgG": "Rubella IgG", "CMVIgG": "CMV IgG", "CMVIgM": "CMV IgM",
    "ToxoIgG": "Toxoplasma IgG", "ToxoIgM": "Toxoplasma IgM",
    "SyphilisScr": "Wife Syphilis Screening", "HusbSyphilis": "Husband Syphilis Screening",
    "GynUltrasound": "Gynecological Ultrasound",
    "HusbDiseaseHis": "Husband Disease History", "HusbBirthDefect": "Husband Birth Defect History",
    "HusbAndrologyDis": "Andrological Disease", "HusbMedication": "Husband Medication History",
    "HusbVaccinated": "Husband Vaccination",
    "HusbFamConsang": "Husband Family Consanguinity", "HusbFamDisease": "Husband Family Disease History",
    "HusbMeatEgg": "Husband Meat/Egg Intake", "HusbVegAverse": "Husband Vegetable Aversion",
    "HusbRawMeat": "Husband Raw Meat Consumption", "HusbSmoke": "Husband Smoking",
    "HusbPassSmoke": "Husband Passive Smoking", "HusbAlcohol": "Husband Alcohol Use",
    "HusbDrugUse": "Husband Drug Use", "HusbEnvExpose": "Husband Environmental Exposure",
    "HusbStress": "Husband Work Stress", "HusbRelatStress": "Husband Relationship Stress",
    "HusbFinance": "Husband Financial Stress", "HusbReady": "Husband Reproductive Readiness",
    "HusbThyroid": "Husband Thyroid Exam", "HusbLung": "Husband Lung Exam",
    "HusbRhythm": "Husband Cardiac Rhythm", "HusbMurmur": "Husband Heart Murmur",
    "HusbLiverSpleen": "Husband Liver/Spleen Exam", "HusbExtremity": "Husband Extremity Exam",
    "HusbAndroExam": "Andrological Exam",
}

# ---- Variable Groups (for Table 1 layout) ----
VAR_GROUPS = {
    "Demographics": [
        "WifeAge", "HusbandAge", "WifeEdu", "HusbandEdu", "WifeOcc", "HusbandOcc",
        "WifeEthnic", "HusbandEthnic", "WifeResident", "HusbandResident"],
    "Wife Physical Examination": [
        "WifeHeight", "WifeBMI", "WifeHR", "WifeSBP", "WifeDBP",
        "WifeMental", "WifeIntel", "WifeFace", "WifePosture", "WifeFaceSpec", "WifeSkinHair",
        "WifeThyroid", "WifeLung", "WifeRhythm", "WifeMurmur",
        "WifeLiverSpleen", "WifeExtremity"],
    "Husband Physical Examination": [
        "HusbHeight", "HusbBMI", "HusbHR", "HusbSBP", "HusbDBP",
        "HusbMental", "HusbIntel", "HusbFace", "HusbPosture", "HusbFaceSpec", "HusbSkinHair",
        "HusbThyroid", "HusbLung", "HusbRhythm", "HusbMurmur",
        "HusbLiverSpleen", "HusbExtremity"],
    "Menstruation & Reproduction": [
        "MenarcheAge", "CycleRegular", "MenstrualVol", "Dysmenorrhea",
        "PrevPregnancy", "Contraception", "WifeGynExam", "GynUltrasound",
        "WifePubHair", "Breast", "Vulva", "GynDisease"],
    "Andrological Examination": [
        "HusbPubHair", "AdamsApple", "Penis", "Foreskin",
        "Testis", "Epididymis", "VasDeferens", "Varicocele",
        "LeftTestVol", "RightTestVol", "HusbAndroExam", "HusbAndrologyDis"],
    "Hematology": [
        "Hemoglobin", "RBC", "Platelet", "WBC", "NeutrophilPct", "LymphocytePct"],
    "Biochemistry & Hormones": [
        "Glucose", "WifeALT", "WifeCreat", "HusbALT", "HusbCreat", "TSH"],
    "Blood Type": [
        "WifeABO", "HusbABO", "WifeRh", "HusbRh"],
    "Infection Screening": [
        "RubellaIgG", "CMVIgG", "CMVIgM", "ToxoIgG", "ToxoIgM",
        "SyphilisScr", "HusbSyphilis", "WifeHepB", "HusbHepB",
        "WifeUrine", "HusbUrine"],
    "Medical & Family History": [
        "WifeDiseaseHx", "WifeBirthDefect", "WifeMedication", "WifeVaccine",
        "WifeFamConsang", "WifeFamDisease", "Consanguinity",
        "HusbDiseaseHis", "HusbBirthDefect", "HusbMedication", "HusbVaccinated",
        "HusbFamConsang", "HusbFamDisease"],
    "Lifestyle (Wife)": [
        "WifeMeatEgg", "WifeVegAverse", "WifeRawMeat",
        "WifeSmoke", "WifePassSmoke", "WifeAlcohol", "WifeDrugUse",
        "WifeHalitosis", "WifeGumBleed"],
    "Lifestyle (Husband)": [
        "HusbMeatEgg", "HusbVegAverse", "HusbRawMeat",
        "HusbSmoke", "HusbPassSmoke", "HusbAlcohol", "HusbDrugUse"],
    "Psychosocial Factors": [
        "WifeEnvExpose", "WifeStress", "WifeRelatStress", "WifeFinance", "WifeReady",
        "HusbEnvExpose", "HusbStress", "HusbRelatStress", "HusbFinance", "HusbReady"],
}

ALL_ANALYSIS_VARS = CONTINUOUS_VARS + ORDINAL_VARS + NOMINAL_VARS + BINARY_VARS

# Dataset display names
DS_TRAINING = "Training"
DS_INTERNAL = "Internal Validation"
DS_TEMPORAL = "Temporal Validation"


# ============================================================
# Helper Functions
# ============================================================
def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def compute_smd(x1, x2):
    m1, m2 = np.nanmean(x1), np.nanmean(x2)
    s1, s2 = np.nanstd(x1, ddof=1), np.nanstd(x2, ddof=1)
    pooled = np.sqrt((s1**2 + s2**2) / 2)
    return abs(m1 - m2) / pooled if pooled > 0 else 0.0


def compute_smd_binary(p1, p2):
    pooled = np.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / 2)
    return abs(p1 - p2) / pooled if pooled > 0 else 0.0


def format_p(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def get_outcome_binary(df):
    if OUTCOME_VAR not in df.columns:
        return pd.Series(dtype=float)
    return df[OUTCOME_VAR].map(
        {True: 1, False: 0, "TRUE": 1, "FALSE": 0, "True": 1, "False": 0, 1: 1, 0: 0}
    ).astype(float)


# ============================================================
# Core Descriptive Statistics Functions
# ============================================================
def describe_continuous(datasets, var):
    results = {}
    all_vals = []
    for name, df in datasets.items():
        if var not in df.columns:
            continue
        vals = safe_numeric(df[var]).dropna()
        all_vals.append(vals)
        n = len(vals)
        miss = len(df) - n
        if n > 0:
            results[name] = {
                "N_obs": n, "Missing": miss, "Missing%": miss / len(df) * 100,
                "Mean": np.mean(vals), "SD": np.std(vals, ddof=1),
                "Median": np.median(vals),
                "Q1": np.percentile(vals, 25), "Q3": np.percentile(vals, 75),
                "Min": np.min(vals), "Max": np.max(vals),
                "Mean_SD": f"{np.mean(vals):.2f} \u00b1 {np.std(vals, ddof=1):.2f}",
                "Median_IQR": f"{np.median(vals):.2f} ({np.percentile(vals, 25):.2f}-{np.percentile(vals, 75):.2f})",
            }
        else:
            results[name] = {
                "N_obs": 0, "Missing": miss, "Missing%": miss / len(df) * 100,
                "Mean": np.nan, "SD": np.nan, "Median": np.nan,
                "Q1": np.nan, "Q3": np.nan, "Min": np.nan, "Max": np.nan,
                "Mean_SD": "—", "Median_IQR": "—",
            }

    p_kw = np.nan
    valid = [v for v in all_vals if len(v) > 0]
    if len(valid) >= 2:
        try:
            _, p_kw = stats.kruskal(*valid)
        except Exception:
            pass

    names = list(datasets.keys())
    smd_int, smd_temp = np.nan, np.nan
    if len(names) >= 2:
        v0 = safe_numeric(datasets[names[0]][var]).dropna() if var in datasets[names[0]].columns else pd.Series()
        v1 = safe_numeric(datasets[names[1]][var]).dropna() if var in datasets[names[1]].columns else pd.Series()
        if len(v0) > 0 and len(v1) > 0:
            smd_int = compute_smd(v0, v1)
    if len(names) >= 3:
        v2 = safe_numeric(datasets[names[2]][var]).dropna() if var in datasets[names[2]].columns else pd.Series()
        if len(v0) > 0 and len(v2) > 0:
            smd_temp = compute_smd(v0, v2)

    return results, p_kw, smd_int, smd_temp


def describe_categorical(datasets, var):
    results = {}
    all_cats = set()
    for name, df in datasets.items():
        if var not in df.columns:
            continue
        s = df[var].copy()
        s = s.map({True: "True", False: "False", 1: "True", 0: "False"}).fillna(s)
        vc = s.value_counts(dropna=True)
        n = vc.sum()
        miss = df[var].isna().sum()
        all_cats.update(vc.index)
        results[name] = {
            "N": n, "Missing": miss,
            "Missing%": miss / len(df) * 100 if len(df) > 0 else 0,
            "counts": vc, "pcts": vc / n * 100 if n > 0 else vc * 0,
        }
    all_cats = sorted(all_cats, key=lambda x: str(x))

    p_chi = np.nan
    names = list(datasets.keys())
    if len(names) >= 2 and len(all_cats) >= 2:
        try:
            ctab = pd.DataFrame(index=all_cats)
            for name in names:
                if name in results:
                    ctab[name] = results[name]["counts"].reindex(all_cats, fill_value=0)
            _, p_chi, _, _ = stats.chi2_contingency(ctab.values)
        except Exception:
            pass

    smd_int, smd_temp = np.nan, np.nan
    if len(all_cats) == 2 and len(names) >= 2:
        true_cat = "True" if "True" in all_cats else all_cats[-1]
        p_vals = []
        for name in names:
            if name in results and results[name]["N"] > 0:
                cnt = results[name]["counts"].get(true_cat, 0)
                p_vals.append(cnt / results[name]["N"])
            else:
                p_vals.append(np.nan)
        if len(p_vals) >= 2 and not np.isnan(p_vals[0]) and not np.isnan(p_vals[1]):
            smd_int = compute_smd_binary(p_vals[0], p_vals[1])
        if len(p_vals) >= 3 and not np.isnan(p_vals[0]) and not np.isnan(p_vals[2]):
            smd_temp = compute_smd_binary(p_vals[0], p_vals[2])

    return results, all_cats, p_chi, smd_int, smd_temp


# ============================================================
# Split Raw Data (identical logic to imputation script)
# ============================================================
def split_raw_data(raw_path):
    print(f"\nReading raw data: {raw_path}")
    if raw_path.endswith(".csv"):
        df_raw = pd.read_csv(raw_path)
    else:
        df_raw = pd.read_excel(raw_path)
    print(f"  Raw data dimensions: {df_raw.shape}")

    df_raw = df_raw.dropna(subset=[OUTCOME_VAR])
    print(f"  After excluding missing outcome: {len(df_raw):,}")

    df_raw[DATE_VAR] = pd.to_datetime(df_raw[DATE_VAR], errors="coerce")
    df_raw["Year"] = df_raw[DATE_VAR].dt.year
    df_raw[OUTCOME_VAR] = df_raw[OUTCOME_VAR].map(
        {True: 1, False: 0, "TRUE": 1, "FALSE": 0, 1: 1, 0: 0})

    df_temporal = df_raw[df_raw["Year"] == 2019].copy()
    df_dev = df_raw[df_raw["Year"].between(2014, 2018) | df_raw["Year"].isnull()].copy()

    df_train, df_internal_val = train_test_split(
        df_dev, test_size=1 - TRAIN_RATIO,
        stratify=df_dev[OUTCOME_VAR].astype(int), random_state=RANDOM_STATE)

    for subset in [df_train, df_internal_val, df_temporal]:
        subset.drop(columns=["Year"], inplace=True, errors="ignore")

    raw_datasets = {
        DS_TRAINING: df_train, DS_INTERNAL: df_internal_val, DS_TEMPORAL: df_temporal}

    print("\n  Pre-imputation dataset split:")
    for name, data in raw_datasets.items():
        y = data[OUTCOME_VAR].astype(int)
        print(f"    {name:25s}: N={len(data):>7,}  |  SA={y.sum():>5,} ({y.mean()*100:.2f}%)")
    return raw_datasets


# ============================================================
# Excel Styles
# ============================================================
class Styles:
    header_font = Font(bold=True, size=11, color="FFFFFF", name="Arial")
    header_fill = PatternFill("solid", fgColor="4472C4")
    section_fill = PatternFill("solid", fgColor="E2EFDA")
    section_font = Font(bold=True, size=11, name="Arial", color="375623")
    normal_font = Font(size=10, name="Arial")
    bold_font = Font(size=10, name="Arial", bold=True)
    sig_font = Font(size=10, name="Arial", bold=True, color="C00000")
    good_font = Font(size=10, name="Arial", bold=True, color="006100")
    italic_font = Font(size=10, name="Arial", italic=True, color="808080")
    note_font = Font(size=9, name="Arial", italic=True, color="666666")
    thin_border = Border(
        left=Side(style="thin", color="BFBFBF"), right=Side(style="thin", color="BFBFBF"),
        top=Side(style="thin", color="BFBFBF"), bottom=Side(style="thin", color="BFBFBF"))
    center = Alignment(horizontal="center")
    wrap_center = Alignment(horizontal="center", wrap_text=True)
    left = Alignment(horizontal="left")

S = Styles()


def write_header(ws, row, headers):
    for c, h in enumerate(headers, 1):
        cell = ws.cell(row=row, column=c, value=h)
        cell.font = S.header_font; cell.fill = S.header_fill
        cell.alignment = S.wrap_center; cell.border = S.thin_border


def wcell(ws, row, col, value, font=None, fill=None, align=None):
    cell = ws.cell(row=row, column=col, value=value)
    cell.font = font or S.normal_font; cell.border = S.thin_border
    cell.alignment = align or S.center
    if fill:
        cell.fill = fill
    return cell


# ============================================================
# Sheet Writers
# ============================================================
def write_overview(wb, raw_ds, imp_ds):
    ws = wb.active
    ws.title = "Dataset Overview"
    ds_names = list(raw_ds.keys())

    headers = ["Metric"]
    for n in ds_names:
        headers += [f"{n} (Pre)", f"{n} (Post)"]
    write_header(ws, 1, headers)

    def add_row(ws, row, label, pre, post):
        wcell(ws, row, 1, label, font=S.bold_font, align=S.left)
        col = 2
        for i in range(len(ds_names)):
            wcell(ws, row, col, pre[i] if i < len(pre) else "—")
            wcell(ws, row, col + 1, post[i] if i < len(post) else "—")
            col += 2

    row = 2
    add_row(ws, row, "Sample Size (N)",
            [f"{len(raw_ds[n]):,}" for n in ds_names],
            [f"{len(imp_ds[n]):,}" if n in imp_ds else "—" for n in ds_names])
    row += 1

    pre_sa, post_sa = [], []
    for n in ds_names:
        y = get_outcome_binary(raw_ds[n])
        pre_sa.append(f"{int(y.sum()):,} ({y.mean()*100:.2f}%)")
        if n in imp_ds:
            y2 = get_outcome_binary(imp_ds[n])
            post_sa.append(f"{int(y2.sum()):,} ({y2.mean()*100:.2f}%)")
        else:
            post_sa.append("—")
    add_row(ws, row, "SA Positive n(%)", pre_sa, post_sa); row += 1

    pre_neg, post_neg = [], []
    for n in ds_names:
        y = get_outcome_binary(raw_ds[n])
        pre_neg.append(f"{int(len(raw_ds[n])-y.sum()):,} ({(1-y.mean())*100:.2f}%)")
        if n in imp_ds:
            y2 = get_outcome_binary(imp_ds[n])
            post_neg.append(f"{int(len(imp_ds[n])-y2.sum()):,} ({(1-y2.mean())*100:.2f}%)")
        else:
            post_neg.append("—")
    add_row(ws, row, "SA Negative n(%)", pre_neg, post_neg); row += 1

    add_row(ws, row, "Total Analysis Variables",
            [str(len([c for c in ALL_ANALYSIS_VARS if c in raw_ds[n].columns])) for n in ds_names],
            [str(len([c for c in ALL_ANALYSIS_VARS if c in imp_ds[n].columns])) if n in imp_ds else "—" for n in ds_names])
    row += 1
    add_row(ws, row, "  Continuous Variables",
            [str(len([c for c in CONTINUOUS_VARS if c in raw_ds[n].columns])) for n in ds_names],
            [str(len([c for c in CONTINUOUS_VARS if c in imp_ds[n].columns])) if n in imp_ds else "—" for n in ds_names])
    row += 1
    add_row(ws, row, "  Categorical Variables",
            [str(len([c for c in CATEGORICAL_VARS if c in raw_ds[n].columns])) for n in ds_names],
            [str(len([c for c in CATEGORICAL_VARS if c in imp_ds[n].columns])) if n in imp_ds else "—" for n in ds_names])
    row += 1

    pre_m, post_m = [], []
    for n in ds_names:
        ac = [c for c in ALL_ANALYSIS_VARS if c in raw_ds[n].columns]
        t = len(raw_ds[n]) * len(ac) if ac else 1
        m = raw_ds[n][ac].isna().sum().sum()
        pre_m.append(f"{int(m):,} ({m/t*100:.2f}%)")
        if n in imp_ds:
            ac2 = [c for c in ALL_ANALYSIS_VARS if c in imp_ds[n].columns]
            t2 = len(imp_ds[n]) * len(ac2) if ac2 else 1
            m2 = imp_ds[n][ac2].isna().sum().sum()
            post_m.append(f"{int(m2):,} ({m2/t2*100:.2f}%)")
        else:
            post_m.append("—")
    add_row(ws, row, "Missing Cells n(%)", pre_m, post_m); row += 1

    pre_mv, post_mv = [], []
    for n in ds_names:
        ac = [c for c in ALL_ANALYSIS_VARS if c in raw_ds[n].columns]
        pre_mv.append(str(sum(1 for c in ac if raw_ds[n][c].isna().any())))
        if n in imp_ds:
            ac2 = [c for c in ALL_ANALYSIS_VARS if c in imp_ds[n].columns]
            post_mv.append(str(sum(1 for c in ac2 if imp_ds[n][c].isna().any())))
        else:
            post_mv.append("—")
    add_row(ws, row, "Variables with Missing", pre_mv, post_mv)

    ws.column_dimensions["A"].width = 28
    for i in range(1, len(headers)):
        ws.column_dimensions[get_column_letter(i + 1)].width = 24
    ws.freeze_panes = "B2"


def write_continuous_sheet(wb, datasets, sheet_name):
    ws = wb.create_sheet(sheet_name)
    ds_names = list(datasets.keys())

    headers = ["Variable", "Label"]
    for n in ds_names:
        headers += [f"{n} N(Miss%)", f"{n} Mean\u00b1SD", f"{n} Median(IQR)", f"{n} Min-Max"]
    headers += ["P (KW)", "SMD(Train vs IntVal)", "SMD(Train vs TempVal)"]
    write_header(ws, 1, headers)

    row = 2
    for var in CONTINUOUS_VARS:
        if not any(var in datasets[n].columns for n in ds_names):
            continue
        res, p_kw, smd_int, smd_temp = describe_continuous(datasets, var)
        wcell(ws, row, 1, var, font=S.bold_font, align=S.left)
        wcell(ws, row, 2, VAR_LABELS.get(var, ""), align=S.left)
        col = 3
        for n in ds_names:
            if n in res:
                r = res[n]
                wcell(ws, row, col, f"{r['N_obs']:,} ({r['Missing%']:.1f}%)")
                wcell(ws, row, col + 1, r["Mean_SD"])
                wcell(ws, row, col + 2, r["Median_IQR"])
                wcell(ws, row, col + 3, f"{r['Min']:.2f}-{r['Max']:.2f}")
            else:
                for o in range(4):
                    wcell(ws, row, col + o, "—")
            col += 4
        pc = wcell(ws, row, col, format_p(p_kw))
        if not np.isnan(p_kw) and p_kw < 0.05: pc.font = S.sig_font
        col += 1
        for sv in [smd_int, smd_temp]:
            c = wcell(ws, row, col, f"{sv:.3f}" if not np.isnan(sv) else "")
            if not np.isnan(sv) and sv > 0.1: c.font = S.sig_font
            col += 1
        row += 1

    ws.column_dimensions["A"].width = 18; ws.column_dimensions["B"].width = 28
    for i in range(2, len(headers)):
        ws.column_dimensions[get_column_letter(i + 1)].width = 24
    ws.freeze_panes = "C2"


def write_categorical_sheet(wb, datasets, sheet_name):
    ws = wb.create_sheet(sheet_name)
    ds_names = list(datasets.keys())
    headers = ["Variable", "Category"]
    for n in ds_names:
        headers += [f"{n} n(%)"]
    headers += ["P (Chi2)", "SMD(Train vs IntVal)", "SMD(Train vs TempVal)"]
    write_header(ws, 1, headers)
    p_col = 3 + len(ds_names)
    row = 2

    for var in BINARY_VARS + ORDINAL_VARS + NOMINAL_VARS:
        if not any(var in datasets[n].columns for n in ds_names):
            continue
        res, cats, p_chi, smd_int, smd_temp = describe_categorical(datasets, var)

        wcell(ws, row, 1, var, font=S.section_font, fill=S.section_fill, align=S.left)
        wcell(ws, row, 2, VAR_LABELS.get(var, ""), font=Font(size=10, name="Arial", italic=True), fill=S.section_fill)
        for cc in range(3, p_col):
            wcell(ws, row, cc, "", fill=S.section_fill)
        pc = wcell(ws, row, p_col, format_p(p_chi))
        if not np.isnan(p_chi) and p_chi < 0.05: pc.font = S.sig_font
        s1c = wcell(ws, row, p_col + 1, f"{smd_int:.3f}" if not np.isnan(smd_int) else "")
        if not np.isnan(smd_int) and smd_int > 0.1: s1c.font = S.sig_font
        s2c = wcell(ws, row, p_col + 2, f"{smd_temp:.3f}" if not np.isnan(smd_temp) else "")
        if not np.isnan(smd_temp) and smd_temp > 0.1: s2c.font = S.sig_font
        row += 1

        # Missing row
        wcell(ws, row, 1, ""); wcell(ws, row, 2, "Missing n(%)", font=S.italic_font)
        for di, n in enumerate(ds_names):
            if n in res:
                mn = res[n]["Missing"]; mp = res[n]["Missing%"]
                v = f"{mn:,} ({mp:.1f}%)" if mn > 0 else "0 (0.0%)"
                wcell(ws, row, 3 + di, v, font=S.sig_font if mn > 0 else S.normal_font)
            else:
                wcell(ws, row, 3 + di, "—")
        for cc in range(p_col, p_col + 3): wcell(ws, row, cc, "")
        row += 1

        for cat in cats:
            wcell(ws, row, 1, "")
            wcell(ws, row, 2, str(cat), align=Alignment(horizontal="left", indent=2))
            for di, n in enumerate(ds_names):
                if n in res and res[n]["N"] > 0:
                    cnt = res[n]["counts"].get(cat, 0)
                    pct = res[n]["pcts"].get(cat, 0)
                    wcell(ws, row, 3 + di, f"{int(cnt):,} ({pct:.1f}%)")
                else:
                    wcell(ws, row, 3 + di, "—")
            for cc in range(p_col, p_col + 3): wcell(ws, row, cc, "")
            row += 1

    ws.column_dimensions["A"].width = 22; ws.column_dimensions["B"].width = 30
    for i in range(len(ds_names)):
        ws.column_dimensions[get_column_letter(3 + i)].width = 22
    for i in range(3):
        ws.column_dimensions[get_column_letter(p_col + i)].width = 22
    ws.freeze_panes = "C2"


def write_missing_comparison(wb, raw_ds, imp_ds):
    ws = wb.create_sheet("Missing Rate Comparison")
    ds_names = list(raw_ds.keys())
    headers = ["Variable", "Type"]
    for n in ds_names:
        headers += [f"{n} Pre n(%)", f"{n} Post n(%)"]
    write_header(ws, 1, headers)
    row = 2
    for var in ALL_ANALYSIS_VARS:
        if not any(var in raw_ds[n].columns for n in ds_names):
            continue
        vtype = ("Continuous" if var in CONTINUOUS_VARS else
                 "Ordinal" if var in ORDINAL_VARS else
                 "Nominal" if var in NOMINAL_VARS else "Binary")
        wcell(ws, row, 1, var, align=S.left); wcell(ws, row, 2, vtype)
        col = 3
        for n in ds_names:
            for dg in [raw_ds, imp_ds]:
                if n in dg and var in dg[n].columns:
                    mn = dg[n][var].isna().sum(); mp = mn / len(dg[n]) * 100
                    v = f"{mn:,} ({mp:.1f}%)"
                    wcell(ws, row, col, v, font=S.sig_font if mp > 0 else S.normal_font)
                else:
                    wcell(ws, row, col, "—")
                col += 1
        row += 1
    ws.column_dimensions["A"].width = 22; ws.column_dimensions["B"].width = 10
    for i in range(2, len(headers)):
        ws.column_dimensions[get_column_letter(i + 1)].width = 22
    ws.freeze_panes = "C2"


def write_imputation_effect(wb, raw_ds, imp_ds):
    ws = wb.create_sheet("Imputation Effect")
    ds_names = list(raw_ds.keys())
    headers = ["Variable", "Type"]
    for n in ds_names:
        headers += [f"{n} Delta Mean", f"{n} Delta SD", f"{n} Delta Median"]
    write_header(ws, 1, headers)
    row = 2

    for var in CONTINUOUS_VARS:
        if not any(var in raw_ds[n].columns for n in ds_names):
            continue
        wcell(ws, row, 1, var, align=S.left); wcell(ws, row, 2, "Continuous")
        col = 3
        for n in ds_names:
            rv = safe_numeric(raw_ds[n][var]).dropna() if (n in raw_ds and var in raw_ds[n].columns) else pd.Series()
            iv = safe_numeric(imp_ds[n][var]).dropna() if (n in imp_ds and var in imp_ds[n].columns) else pd.Series()
            if len(rv) > 0 and len(iv) > 0:
                wcell(ws, row, col, f"{np.mean(iv)-np.mean(rv):+.3f}")
                wcell(ws, row, col+1, f"{np.std(iv,ddof=1)-np.std(rv,ddof=1):+.3f}")
                wcell(ws, row, col+2, f"{np.median(iv)-np.median(rv):+.3f}")
            else:
                for o in range(3): wcell(ws, row, col+o, "—")
            col += 3
        row += 1

    row += 1
    wcell(ws, row, 1, "Binary Variables (Delta True%, pp)", font=S.section_font, fill=S.section_fill, align=S.left)
    for c in range(2, 3 + len(ds_names)*3): wcell(ws, row, c, "", fill=S.section_fill)
    row += 1

    bh = ["Variable", "Type"]
    for n in ds_names: bh += [f"{n} Delta True%", "", ""]
    write_header(ws, row, bh); row += 1

    for var in BINARY_VARS:
        if not any(var in raw_ds[n].columns for n in ds_names):
            continue
        wcell(ws, row, 1, var, align=S.left); wcell(ws, row, 2, "Binary")
        col = 3
        for n in ds_names:
            rs = raw_ds[n][var] if (n in raw_ds and var in raw_ds[n].columns) else pd.Series()
            ims = imp_ds[n][var] if (n in imp_ds and var in imp_ds[n].columns) else pd.Series()
            rm = rs.map({True:1,False:0,"TRUE":1,"FALSE":0,"True":1,"False":0,1:1,0:0})
            im = ims.map({True:1,False:0,"TRUE":1,"FALSE":0,"True":1,"False":0,1:1,0:0})
            rp = rm.dropna().mean()*100 if rm.dropna().shape[0]>0 else np.nan
            ip = im.dropna().mean()*100 if im.dropna().shape[0]>0 else np.nan
            if not np.isnan(rp) and not np.isnan(ip):
                wcell(ws, row, col, f"{ip-rp:+.2f}pp")
            else:
                wcell(ws, row, col, "—")
            wcell(ws, row, col+1, ""); wcell(ws, row, col+2, "")
            col += 3
        row += 1

    ws.column_dimensions["A"].width = 22; ws.column_dimensions["B"].width = 10
    for i in range(2, 3+len(ds_names)*3):
        ws.column_dimensions[get_column_letter(i+1)].width = 16
    ws.freeze_panes = "C2"


def write_smd_sheet(wb, imp_ds):
    ws = wb.create_sheet("SMD Summary")
    ds_names = list(imp_ds.keys())
    headers = ["Variable", "Type", "SMD (Train vs IntVal)", "SMD (Train vs TempVal)", "Balanced (<0.1)?"]
    write_header(ws, 1, headers)
    row = 2; records = []

    for var in CONTINUOUS_VARS:
        if not any(var in imp_ds[n].columns for n in ds_names): continue
        _, _, si, st = describe_continuous(imp_ds, var)
        records.append((var, "Continuous", si, st))
    for var in BINARY_VARS:
        if not any(var in imp_ds[n].columns for n in ds_names): continue
        _, _, _, si, st = describe_categorical(imp_ds, var)
        records.append((var, "Binary", si, st))

    for var, vt, s1, s2 in records:
        wcell(ws, row, 1, var, align=S.left); wcell(ws, row, 2, vt)
        for ci, sv in [(3,s1),(4,s2)]:
            c = wcell(ws, row, ci, f"{sv:.4f}" if not np.isnan(sv) else "")
            if not np.isnan(sv) and sv > 0.1: c.font = S.sig_font
        bal = "Yes" if (not np.isnan(s1) and s1<=0.1 and not np.isnan(s2) and s2<=0.1) else "No"
        if np.isnan(s1) or np.isnan(s2): bal = "N/A"
        wcell(ws, row, 5, bal, font=S.good_font if bal=="Yes" else S.sig_font)
        row += 1

    row += 1; wcell(ws, row, 1, "Summary", font=S.bold_font, align=S.left); row += 1
    nb = sum(1 for _,_,s1,s2 in records if not np.isnan(s1) and s1<=0.1 and not np.isnan(s2) and s2<=0.1)
    nt = sum(1 for _,_,s1,s2 in records if not np.isnan(s1) and not np.isnan(s2))
    txt = f"SMD <= 0.1: {nb}/{nt} ({nb/nt*100:.1f}%)" if nt > 0 else "N/A"
    wcell(ws, row, 1, txt, align=S.left)

    ws.column_dimensions["A"].width = 22; ws.column_dimensions["B"].width = 12
    for i in range(3,6): ws.column_dimensions[get_column_letter(i)].width = 24
    ws.freeze_panes = "A2"


def write_table1(wb, datasets, sheet_name):
    """Table 1 style: grouped continuous + categorical in one sheet."""
    ws = wb.create_sheet(sheet_name)
    ds_names = list(datasets.keys())
    headers = ["Group", "Variable", "Label"]
    for n in ds_names:
        headers += [f"{n} (N={len(datasets[n]):,})"]
    headers += ["P-value", "Test"]
    write_header(ws, 1, headers)
    p_col = 4 + len(ds_names); test_col = p_col + 1
    row = 2

    for gname, gvars in VAR_GROUPS.items():
        wcell(ws, row, 1, gname, font=S.section_font, fill=S.section_fill, align=S.left)
        for c in range(2, test_col+1): wcell(ws, row, c, "", fill=S.section_fill)
        row += 1

        for var in gvars:
            if not any(var in datasets[n].columns for n in ds_names):
                continue
            label = VAR_LABELS.get(var, "")

            if var in CONTINUOUS_VARS:
                res, p_kw, _, _ = describe_continuous(datasets, var)
                wcell(ws, row, 1, ""); wcell(ws, row, 2, var, font=S.bold_font, align=S.left)
                wcell(ws, row, 3, label, align=S.left)
                col = 4
                for n in ds_names:
                    wcell(ws, row, col, res[n]["Median_IQR"] if n in res else "—")
                    col += 1
                pc = wcell(ws, row, p_col, format_p(p_kw))
                if not np.isnan(p_kw) and p_kw < 0.05: pc.font = S.sig_font
                wcell(ws, row, test_col, "KW"); row += 1

            elif var in BINARY_VARS:
                res, cats, p_chi, _, _ = describe_categorical(datasets, var)
                true_cat = "True" if "True" in cats else (cats[-1] if cats else None)
                wcell(ws, row, 1, ""); wcell(ws, row, 2, var, font=S.bold_font, align=S.left)
                wcell(ws, row, 3, label, align=S.left)
                col = 4
                for n in ds_names:
                    if n in res and res[n]["N"]>0 and true_cat:
                        cnt = res[n]["counts"].get(true_cat, 0)
                        pct = res[n]["pcts"].get(true_cat, 0)
                        wcell(ws, row, col, f"{int(cnt):,} ({pct:.1f}%)")
                    else:
                        wcell(ws, row, col, "—")
                    col += 1
                pc = wcell(ws, row, p_col, format_p(p_chi))
                if not np.isnan(p_chi) and p_chi < 0.05: pc.font = S.sig_font
                wcell(ws, row, test_col, "Chi2"); row += 1

            elif var in ORDINAL_VARS + NOMINAL_VARS:
                res, cats, p_chi, _, _ = describe_categorical(datasets, var)
                wcell(ws, row, 1, ""); wcell(ws, row, 2, var, font=S.bold_font, align=S.left)
                wcell(ws, row, 3, label, align=S.left)
                for c in range(4, 4+len(ds_names)): wcell(ws, row, c, "")
                pc = wcell(ws, row, p_col, format_p(p_chi))
                if not np.isnan(p_chi) and p_chi < 0.05: pc.font = S.sig_font
                wcell(ws, row, test_col, "Chi2"); row += 1
                for cat in cats:
                    wcell(ws, row, 1, ""); wcell(ws, row, 2, "")
                    wcell(ws, row, 3, f"  {cat}", align=S.left)
                    col = 4
                    for n in ds_names:
                        if n in res and res[n]["N"]>0:
                            cnt = res[n]["counts"].get(cat, 0)
                            pct = res[n]["pcts"].get(cat, 0)
                            wcell(ws, row, col, f"{int(cnt):,} ({pct:.1f}%)")
                        else:
                            wcell(ws, row, col, "—")
                        col += 1
                    wcell(ws, row, p_col, ""); wcell(ws, row, test_col, "")
                    row += 1

    row += 1
    note = ("Note: Continuous variables presented as Median(IQR), P-value from Kruskal-Wallis test; "
            "Binary variables presented as n(%) for positive cases; "
            "Multi-category variables presented as n(%) per level, P-value from Chi-square test.")
    wcell(ws, row, 1, note, font=S.note_font, align=S.left)
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=test_col)

    ws.column_dimensions["A"].width = 20; ws.column_dimensions["B"].width = 22
    ws.column_dimensions["C"].width = 30
    for i in range(len(ds_names)):
        ws.column_dimensions[get_column_letter(4+i)].width = 26
    ws.column_dimensions[get_column_letter(p_col)].width = 12
    ws.column_dimensions[get_column_letter(test_col)].width = 8
    ws.freeze_panes = "D2"


def write_variable_dictionary(wb, raw_ds):
    ws = wb.create_sheet("Variable Dictionary")
    headers = ["No.", "Variable Name", "Label", "Variable Type", "Group", "Value Range / Description"]
    write_header(ws, 1, headers)

    var_to_group = {}
    for gn, gv in VAR_GROUPS.items():
        for v in gv:
            var_to_group[v] = gn

    ref_df = next(iter(raw_ds.values()), None)
    row = 2; seq = 1

    # Outcome
    wcell(ws, row, 1, "", fill=S.section_fill)
    wcell(ws, row, 2, "-- Outcome Variable --", font=S.section_font, fill=S.section_fill, align=S.left)
    for c in range(3, 7): wcell(ws, row, c, "", fill=S.section_fill)
    row += 1
    wcell(ws, row, 1, seq); wcell(ws, row, 2, OUTCOME_VAR, font=S.bold_font, align=S.left)
    wcell(ws, row, 3, VAR_LABELS.get(OUTCOME_VAR, ""), align=S.left)
    wcell(ws, row, 4, "Binary (Outcome)"); wcell(ws, row, 5, "—")
    wcell(ws, row, 6, "True = Spontaneous Abortion, False = No Abortion", align=S.left)
    seq += 1; row += 1

    type_sections = [
        ("-- Continuous Variables --", CONTINUOUS_VARS, "Continuous"),
        ("-- Ordinal Variables --", ORDINAL_VARS, "Ordinal"),
        ("-- Nominal Variables --", NOMINAL_VARS, "Nominal"),
        ("-- Binary Variables --", BINARY_VARS, "Binary"),
    ]

    for stitle, vlist, vtype in type_sections:
        wcell(ws, row, 1, "", fill=S.section_fill)
        wcell(ws, row, 2, stitle, font=S.section_font, fill=S.section_fill, align=S.left)
        for c in range(3, 7): wcell(ws, row, c, "", fill=S.section_fill)
        row += 1

        for var in vlist:
            wcell(ws, row, 1, seq); wcell(ws, row, 2, var, font=S.bold_font, align=S.left)
            wcell(ws, row, 3, VAR_LABELS.get(var, ""), align=S.left)
            wcell(ws, row, 4, vtype)
            wcell(ws, row, 5, var_to_group.get(var, "Other"), align=S.left)
            rdesc = ""
            if ref_df is not None and var in ref_df.columns:
                if vtype == "Continuous":
                    vals = safe_numeric(ref_df[var]).dropna()
                    if len(vals) > 0:
                        rdesc = f"Range: {vals.min():.1f} ~ {vals.max():.1f}"
                elif vtype == "Binary":
                    rdesc = "True / False"
                else:
                    uniq = ref_df[var].dropna().unique()
                    if len(uniq) <= 10:
                        rdesc = ", ".join(sorted([str(x) for x in uniq]))
                    else:
                        rdesc = f"{len(uniq)} categories"
            wcell(ws, row, 6, rdesc, align=S.left)
            seq += 1; row += 1

    ws.column_dimensions["A"].width = 6; ws.column_dimensions["B"].width = 24
    ws.column_dimensions["C"].width = 32; ws.column_dimensions["D"].width = 16
    ws.column_dimensions["E"].width = 28; ws.column_dimensions["F"].width = 50
    ws.freeze_panes = "A2"


# ============================================================
# Main Report Generator
# ============================================================
def create_full_report(raw_ds, imp_ds, output_path):
    wb = Workbook()
    print("\nGenerating Excel report...")
    print("  Sheet 1:  Dataset Overview");        write_overview(wb, raw_ds, imp_ds)
    print("  Sheet 2:  Pre-Imputation Table 1");  write_table1(wb, raw_ds, "Pre-Imp Table 1")
    print("  Sheet 3:  Post-Imputation Table 1"); write_table1(wb, imp_ds, "Post-Imp Table 1")
    print("  Sheet 4:  Pre-Imp Continuous");       write_continuous_sheet(wb, raw_ds, "Pre-Imp Continuous")
    print("  Sheet 5:  Post-Imp Continuous");      write_continuous_sheet(wb, imp_ds, "Post-Imp Continuous")
    print("  Sheet 6:  Pre-Imp Categorical");      write_categorical_sheet(wb, raw_ds, "Pre-Imp Categorical")
    print("  Sheet 7:  Post-Imp Categorical");     write_categorical_sheet(wb, imp_ds, "Post-Imp Categorical")
    print("  Sheet 8:  Missing Rate Comparison");  write_missing_comparison(wb, raw_ds, imp_ds)
    print("  Sheet 9:  Imputation Effect");        write_imputation_effect(wb, raw_ds, imp_ds)
    print("  Sheet 10: SMD Summary");              write_smd_sheet(wb, imp_ds)
    print("  Sheet 11: Variable Dictionary");      write_variable_dictionary(wb, raw_ds)
    wb.save(output_path)
    print(f"\nReport saved: {output_path}")


def print_console_summary(raw_ds, imp_ds):
    print("\n" + "=" * 100)
    print("DESCRIPTIVE STATISTICS SUMMARY (Pre- vs Post-Imputation)")
    print("=" * 100)
    ds_names = list(raw_ds.keys())
    print(f"\n{'Dataset':<28} {'N':>8}  {'SA%(Pre)':>10}  {'SA%(Post)':>10}  {'Miss%(Pre)':>11}  {'Miss%(Post)':>11}")
    print("-" * 90)
    for n in ds_names:
        N = len(raw_ds[n]); yr = get_outcome_binary(raw_ds[n])
        ac = [c for c in ALL_ANALYSIS_VARS if c in raw_ds[n].columns]
        tr = len(raw_ds[n])*len(ac) if ac else 1
        mr = raw_ds[n][ac].isna().sum().sum()/tr*100
        if n in imp_ds:
            yi = get_outcome_binary(imp_ds[n])
            ac2 = [c for c in ALL_ANALYSIS_VARS if c in imp_ds[n].columns]
            ti = len(imp_ds[n])*len(ac2) if ac2 else 1
            mi = imp_ds[n][ac2].isna().sum().sum()/ti*100
            sa_i = f"{yi.mean()*100:.2f}%"; mi_s = f"{mi:.2f}%"
        else:
            sa_i = "—"; mi_s = "—"
        print(f"  {n:<26} {N:>8,}  {yr.mean()*100:>9.2f}%  {sa_i:>10}  {mr:>10.2f}%  {mi_s:>11}")

    print(f"\nContinuous variable examples (Pre -> Post Mean\u00b1SD):")
    count = 0
    for var in CONTINUOUS_VARS:
        if count >= 5: break
        if not any(var in raw_ds[n].columns for n in ds_names): continue
        rr, _, _, _ = describe_continuous(raw_ds, var)
        ri, _, _, _ = describe_continuous(imp_ds, var)
        print(f"  {var} ({VAR_LABELS.get(var, '')}):")
        for n in ds_names:
            rs = rr[n]["Mean_SD"] if n in rr else "—"
            iss = ri[n]["Mean_SD"] if n in ri else "—"
            print(f"    {n}: {rs}  ->  {iss}")
        count += 1
    print(f"\n  ... (see Excel report for full results)")
    print("=" * 100)


# ============================================================
# Entry Point
# ============================================================
if __name__ == "__main__":

    # ---- File Path Configuration ----
    RAW_DATA_PATH = "./Data/dataset1008.csv"
    TRAIN_IMP_PATH = "./Data/data_training_imputed.csv"
    INTVAL_IMP_PATH = "./Data/data_internal_validation_imputed.csv"
    TEMPVAL_IMP_PATH = "./Data/data_temporal_validation_imputed.csv"
    OUTPUT_PATH = "./Data/statistical_description.xlsx"

    # ---- Step 1: Split raw data into pre-imputation subsets ----
    if os.path.exists(RAW_DATA_PATH):
        raw_datasets = split_raw_data(RAW_DATA_PATH)
    else:
        print(f"\nWarning: Raw data file not found: {RAW_DATA_PATH}")
        print("  Will only describe post-imputation data.")
        raw_datasets = {}

    # ---- Step 2: Read post-imputation data ----
    print("\nReading post-imputation data...")
    imp_datasets = {}
    for path, name in [(TRAIN_IMP_PATH, DS_TRAINING),
                        (INTVAL_IMP_PATH, DS_INTERNAL),
                        (TEMPVAL_IMP_PATH, DS_TEMPORAL)]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            imp_datasets[name] = df
            print(f"  {name}: {path} ({len(df):,} rows)")
        else:
            print(f"  Warning: File not found: {path}")

    if not imp_datasets:
        ALL_PATH = "data_all_imputed_with_splits.csv"
        if os.path.exists(ALL_PATH):
            print(f"\nReading from combined file: {ALL_PATH}")
            df_all = pd.read_csv(ALL_PATH)
            for label, tag in [(DS_TRAINING, "Training"),
                               (DS_INTERNAL, "Internal_Validation"),
                               (DS_TEMPORAL, "Temporal_Validation")]:
                sub = df_all[df_all["Dataset"] == tag]
                if len(sub) > 0:
                    imp_datasets[label] = sub.drop(columns=["Dataset"], errors="ignore")
                    print(f"  {label}: {len(sub):,} rows")

    # ---- Fallback ----
    if not imp_datasets and not raw_datasets:
        print("\nError: No data files found! Check path configuration.")
        sys.exit(1)
    if not imp_datasets:
        print("\nWarning: Post-imputation data not found. Describing pre-imputation only.")
        imp_datasets = raw_datasets
    if not raw_datasets:
        print("\nWarning: Raw data not found. Describing post-imputation only.")
        raw_datasets = imp_datasets

    # ---- Step 3: Output ----
    print_console_summary(raw_datasets, imp_datasets)
    os.makedirs(os.path.dirname(OUTPUT_PATH) if os.path.dirname(OUTPUT_PATH) else ".", exist_ok=True)
    create_full_report(raw_datasets, imp_datasets, OUTPUT_PATH)

    print(f"\nDone!")
    print(f"  Excel report: {OUTPUT_PATH}")
    print(f"  11 Sheets: Overview / Pre-Imp Table1 / Post-Imp Table1 /")
    print(f"    Pre-Imp Continuous / Post-Imp Continuous / Pre-Imp Categorical / Post-Imp Categorical /")
    print(f"    Missing Rate Comparison / Imputation Effect / SMD Summary / Variable Dictionary")


Reading raw data: ./Data/dataset1008.csv
  Raw data dimensions: (402226, 137)
  After excluding missing outcome: 402,226

  Pre-imputation dataset split:
    Training                 : N=360,363  |  SA=11,753 (3.26%)
    Internal Validation      : N= 40,041  |  SA=1,306 (3.26%)
    Temporal Validation      : N=  1,822  |  SA=  262 (14.38%)

Reading post-imputation data...
  Training: ./Data/data_training_imputed.csv (360,363 rows)
  Internal Validation: ./Data/data_internal_validation_imputed.csv (40,041 rows)
  Temporal Validation: ./Data/data_temporal_validation_imputed.csv (1,822 rows)

DESCRIPTIVE STATISTICS SUMMARY (Pre- vs Post-Imputation)

Dataset                             N    SA%(Pre)   SA%(Post)   Miss%(Pre)  Miss%(Post)
------------------------------------------------------------------------------------------
  Training                    360,363       3.26%       3.26%        4.19%        0.00%
  Internal Validation          40,041       3.26%       3.26%        4.19%   

In [4]:
"""
Statistical Description Script (Pre- and Post-Imputation)
===========================================================
Comprehensive descriptive statistics for three datasets (Training, Internal
Validation, Temporal Validation) before and after imputation.

Excel Report Sheets:
  1.  Dataset Overview         — Sample sizes, outcome distribution, missingness
  2.  Pre-Imputation Table 1   — Table 1 style: grouped continuous + categorical
  3.  Post-Imputation Table 1  — Same, post-imputation
  4.  Pre-Imp Continuous Vars  — Mean±SD, Median(IQR), Min-Max, KW test, SMD
  5.  Post-Imp Continuous Vars — Same
  6.  Pre-Imp Categorical Vars — n(%) per category, chi-square test, SMD
  7.  Post-Imp Categorical Vars— Same
  8.  Missing Rate Comparison  — Per-variable missingness before vs after
  9.  Imputation Effect         — Delta Mean/SD for continuous, Delta % for binary
  10. SMD Summary              — Post-imputation balance: Training vs Validation
  11. Variable Dictionary      — Variable name, label, type, group, value range

Dependencies:
  pip install pandas numpy scipy openpyxl scikit-learn
"""

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import train_test_split
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import warnings
import sys
import os

warnings.filterwarnings("ignore")

# ============================================================
# Global Configuration
# ============================================================
OUTCOME_VAR = "SpontAbortion"
INDEX_VAR = "Index"
DATE_VAR = "BaseInfoDate"
RANDOM_STATE = 42
TRAIN_RATIO = 0.9

# ============================================================
# Variable Definitions
# ============================================================
CONTINUOUS_VARS = [
    "WifeAge", "HusbandAge", "MenarcheAge",
    "WifeHeight", "WifeBMI", "WifeHR", "WifeSBP", "WifeDBP",
    "HusbHeight", "HusbBMI", "HusbHR", "HusbSBP", "HusbDBP",
    "Hemoglobin", "RBC", "Platelet", "WBC",
    "NeutrophilPct", "LymphocytePct",
    "Glucose", "WifeALT", "WifeCreat", "TSH",
    "HusbALT", "HusbCreat",
    "LeftTestVol", "RightTestVol",
]

ORDINAL_VARS = ["WifeEdu", "HusbandEdu", "WifeOcc", "HusbandOcc"]

NOMINAL_VARS = [
    "WifeEthnic", "HusbandEthnic",
    "WifeResident", "HusbandResident",
    "WifeABO", "HusbABO",
    "CycleRegular", "MenstrualVol", "Dysmenorrhea",
    "WifeMental", "WifeIntel", "WifeFace", "WifePosture",
    "WifeFaceSpec", "WifeSkinHair",
    "HusbMental", "HusbIntel", "HusbFace", "HusbPosture",
    "HusbFaceSpec", "HusbSkinHair",
    "WifePubHair", "Breast", "Vulva",
    "HusbPubHair", "AdamsApple", "Penis", "Foreskin",
    "Testis", "Epididymis", "VasDeferens", "Varicocele",
    "WifeUrine", "HusbUrine",
    "WifeHepB", "HusbHepB",
]

BINARY_VARS = [
    "WifeDiseaseHx", "WifeBirthDefect", "GynDisease",
    "WifeMedication", "WifeVaccine", "Contraception",
    "PrevPregnancy", "Consanguinity", "WifeFamConsang", "WifeFamDisease",
    "WifeMeatEgg", "WifeVegAverse", "WifeRawMeat",
    "WifeSmoke", "WifePassSmoke", "WifeAlcohol",
    "WifeDrugUse", "WifeHalitosis", "WifeGumBleed",
    "WifeEnvExpose", "WifeStress", "WifeRelatStress",
    "WifeFinance", "WifeReady",
    "WifeThyroid", "WifeLung", "WifeRhythm", "WifeMurmur",
    "WifeLiverSpleen", "WifeExtremity", "WifeGynExam",
    "WifeRh", "HusbRh",
    "RubellaIgG", "CMVIgG", "CMVIgM",
    "ToxoIgG", "ToxoIgM",
    "SyphilisScr", "HusbSyphilis",
    "GynUltrasound",
    "HusbDiseaseHis", "HusbBirthDefect", "HusbAndrologyDis",
    "HusbMedication", "HusbVaccinated",
    "HusbFamConsang", "HusbFamDisease",
    "HusbMeatEgg", "HusbVegAverse", "HusbRawMeat",
    "HusbSmoke", "HusbPassSmoke", "HusbAlcohol", "HusbDrugUse",
    "HusbEnvExpose", "HusbStress", "HusbRelatStress",
    "HusbFinance", "HusbReady",
    "HusbThyroid", "HusbLung", "HusbRhythm", "HusbMurmur",
    "HusbLiverSpleen", "HusbExtremity", "HusbAndroExam",
]

CATEGORICAL_VARS = ORDINAL_VARS + NOMINAL_VARS + BINARY_VARS

# ---- Variable Labels (English) ----
VAR_LABELS = {
    # Outcome
    "SpontAbortion": "Spontaneous Abortion",
    # Continuous
    "WifeAge": "Wife Age (years)", "HusbandAge": "Husband Age (years)",
    "MenarcheAge": "Menarche Age (years)",
    "WifeHeight": "Wife Height (cm)", "WifeBMI": "Wife BMI (kg/m2)",
    "WifeHR": "Wife Heart Rate (bpm)", "WifeSBP": "Wife SBP (mmHg)", "WifeDBP": "Wife DBP (mmHg)",
    "HusbHeight": "Husband Height (cm)", "HusbBMI": "Husband BMI (kg/m2)",
    "HusbHR": "Husband Heart Rate (bpm)", "HusbSBP": "Husband SBP (mmHg)", "HusbDBP": "Husband DBP (mmHg)",
    "Hemoglobin": "Hemoglobin (g/L)", "RBC": "Red Blood Cells (x10^12/L)",
    "Platelet": "Platelet Count (x10^9/L)", "WBC": "White Blood Cells (x10^9/L)",
    "NeutrophilPct": "Neutrophil Percentage (%)", "LymphocytePct": "Lymphocyte Percentage (%)",
    "Glucose": "Fasting Glucose (mmol/L)", "WifeALT": "Wife ALT (U/L)", "WifeCreat": "Wife Creatinine (umol/L)",
    "TSH": "Thyroid Stimulating Hormone (mIU/L)",
    "HusbALT": "Husband ALT (U/L)", "HusbCreat": "Husband Creatinine (umol/L)",
    "LeftTestVol": "Left Testicular Volume (ml)", "RightTestVol": "Right Testicular Volume (ml)",
    # Ordinal
    "WifeEdu": "Wife Education Level", "HusbandEdu": "Husband Education Level",
    "WifeOcc": "Wife Occupation", "HusbandOcc": "Husband Occupation",
    # Nominal
    "WifeEthnic": "Wife Ethnicity", "HusbandEthnic": "Husband Ethnicity",
    "WifeResident": "Wife Residency", "HusbandResident": "Husband Residency",
    "WifeABO": "Wife ABO Blood Type", "HusbABO": "Husband ABO Blood Type",
    "CycleRegular": "Menstrual Cycle Regularity", "MenstrualVol": "Menstrual Volume",
    "Dysmenorrhea": "Dysmenorrhea",
    "WifeMental": "Wife Mental Status", "WifeIntel": "Wife Intelligence",
    "WifeFace": "Wife Facial Appearance", "WifePosture": "Wife Posture",
    "WifeFaceSpec": "Wife Facial Features", "WifeSkinHair": "Wife Skin & Hair",
    "HusbMental": "Husband Mental Status", "HusbIntel": "Husband Intelligence",
    "HusbFace": "Husband Facial Appearance", "HusbPosture": "Husband Posture",
    "HusbFaceSpec": "Husband Facial Features", "HusbSkinHair": "Husband Skin & Hair",
    "WifePubHair": "Female Pubic Hair", "Breast": "Breast Development", "Vulva": "Vulva",
    "HusbPubHair": "Male Pubic Hair", "AdamsApple": "Adam's Apple", "Penis": "Penis",
    "Foreskin": "Foreskin", "Testis": "Testis", "Epididymis": "Epididymis",
    "VasDeferens": "Vas Deferens", "Varicocele": "Varicocele",
    "WifeUrine": "Wife Urinalysis", "HusbUrine": "Husband Urinalysis",
    "WifeHepB": "Wife Hepatitis B", "HusbHepB": "Husband Hepatitis B",
    # Binary
    "WifeDiseaseHx": "Wife Disease History", "WifeBirthDefect": "Wife Birth Defect History",
    "GynDisease": "Gynecological Disease", "WifeMedication": "Wife Medication History",
    "WifeVaccine": "Wife Vaccination", "Contraception": "Contraception History",
    "PrevPregnancy": "Previous Pregnancy", "Consanguinity": "Consanguineous Marriage",
    "WifeFamConsang": "Wife Family Consanguinity", "WifeFamDisease": "Wife Family Disease History",
    "WifeMeatEgg": "Wife Meat/Egg Intake", "WifeVegAverse": "Wife Vegetable Aversion",
    "WifeRawMeat": "Wife Raw Meat Consumption", "WifeSmoke": "Wife Smoking",
    "WifePassSmoke": "Wife Passive Smoking", "WifeAlcohol": "Wife Alcohol Use",
    "WifeDrugUse": "Wife Drug Use", "WifeHalitosis": "Wife Halitosis",
    "WifeGumBleed": "Wife Gum Bleeding", "WifeEnvExpose": "Wife Environmental Exposure",
    "WifeStress": "Wife Work Stress", "WifeRelatStress": "Wife Relationship Stress",
    "WifeFinance": "Wife Financial Stress", "WifeReady": "Wife Reproductive Readiness",
    "WifeThyroid": "Wife Thyroid Exam", "WifeLung": "Wife Lung Exam",
    "WifeRhythm": "Wife Cardiac Rhythm", "WifeMurmur": "Wife Heart Murmur",
    "WifeLiverSpleen": "Wife Liver/Spleen Exam", "WifeExtremity": "Wife Extremity Exam",
    "WifeGynExam": "Gynecological Exam", "WifeRh": "Wife Rh Blood Type", "HusbRh": "Husband Rh Blood Type",
    "RubellaIgG": "Rubella IgG", "CMVIgG": "CMV IgG", "CMVIgM": "CMV IgM",
    "ToxoIgG": "Toxoplasma IgG", "ToxoIgM": "Toxoplasma IgM",
    "SyphilisScr": "Wife Syphilis Screening", "HusbSyphilis": "Husband Syphilis Screening",
    "GynUltrasound": "Gynecological Ultrasound",
    "HusbDiseaseHis": "Husband Disease History", "HusbBirthDefect": "Husband Birth Defect History",
    "HusbAndrologyDis": "Andrological Disease", "HusbMedication": "Husband Medication History",
    "HusbVaccinated": "Husband Vaccination",
    "HusbFamConsang": "Husband Family Consanguinity", "HusbFamDisease": "Husband Family Disease History",
    "HusbMeatEgg": "Husband Meat/Egg Intake", "HusbVegAverse": "Husband Vegetable Aversion",
    "HusbRawMeat": "Husband Raw Meat Consumption", "HusbSmoke": "Husband Smoking",
    "HusbPassSmoke": "Husband Passive Smoking", "HusbAlcohol": "Husband Alcohol Use",
    "HusbDrugUse": "Husband Drug Use", "HusbEnvExpose": "Husband Environmental Exposure",
    "HusbStress": "Husband Work Stress", "HusbRelatStress": "Husband Relationship Stress",
    "HusbFinance": "Husband Financial Stress", "HusbReady": "Husband Reproductive Readiness",
    "HusbThyroid": "Husband Thyroid Exam", "HusbLung": "Husband Lung Exam",
    "HusbRhythm": "Husband Cardiac Rhythm", "HusbMurmur": "Husband Heart Murmur",
    "HusbLiverSpleen": "Husband Liver/Spleen Exam", "HusbExtremity": "Husband Extremity Exam",
    "HusbAndroExam": "Andrological Exam",
}

# ---- Variable Groups (for Table 1 layout) ----
VAR_GROUPS = {
    "Demographics": [
        "WifeAge", "HusbandAge", "WifeEdu", "HusbandEdu", "WifeOcc", "HusbandOcc",
        "WifeEthnic", "HusbandEthnic", "WifeResident", "HusbandResident"],
    "Wife Physical Examination": [
        "WifeHeight", "WifeBMI", "WifeHR", "WifeSBP", "WifeDBP",
        "WifeMental", "WifeIntel", "WifeFace", "WifePosture", "WifeFaceSpec", "WifeSkinHair",
        "WifeThyroid", "WifeLung", "WifeRhythm", "WifeMurmur",
        "WifeLiverSpleen", "WifeExtremity"],
    "Husband Physical Examination": [
        "HusbHeight", "HusbBMI", "HusbHR", "HusbSBP", "HusbDBP",
        "HusbMental", "HusbIntel", "HusbFace", "HusbPosture", "HusbFaceSpec", "HusbSkinHair",
        "HusbThyroid", "HusbLung", "HusbRhythm", "HusbMurmur",
        "HusbLiverSpleen", "HusbExtremity"],
    "Menstruation & Reproduction": [
        "MenarcheAge", "CycleRegular", "MenstrualVol", "Dysmenorrhea",
        "PrevPregnancy", "Contraception", "WifeGynExam", "GynUltrasound",
        "WifePubHair", "Breast", "Vulva", "GynDisease"],
    "Andrological Examination": [
        "HusbPubHair", "AdamsApple", "Penis", "Foreskin",
        "Testis", "Epididymis", "VasDeferens", "Varicocele",
        "LeftTestVol", "RightTestVol", "HusbAndroExam", "HusbAndrologyDis"],
    "Hematology": [
        "Hemoglobin", "RBC", "Platelet", "WBC", "NeutrophilPct", "LymphocytePct"],
    "Biochemistry & Hormones": [
        "Glucose", "WifeALT", "WifeCreat", "HusbALT", "HusbCreat", "TSH"],
    "Blood Type": [
        "WifeABO", "HusbABO", "WifeRh", "HusbRh"],
    "Infection Screening": [
        "RubellaIgG", "CMVIgG", "CMVIgM", "ToxoIgG", "ToxoIgM",
        "SyphilisScr", "HusbSyphilis", "WifeHepB", "HusbHepB",
        "WifeUrine", "HusbUrine"],
    "Medical & Family History": [
        "WifeDiseaseHx", "WifeBirthDefect", "WifeMedication", "WifeVaccine",
        "WifeFamConsang", "WifeFamDisease", "Consanguinity",
        "HusbDiseaseHis", "HusbBirthDefect", "HusbMedication", "HusbVaccinated",
        "HusbFamConsang", "HusbFamDisease"],
    "Lifestyle (Wife)": [
        "WifeMeatEgg", "WifeVegAverse", "WifeRawMeat",
        "WifeSmoke", "WifePassSmoke", "WifeAlcohol", "WifeDrugUse",
        "WifeHalitosis", "WifeGumBleed"],
    "Lifestyle (Husband)": [
        "HusbMeatEgg", "HusbVegAverse", "HusbRawMeat",
        "HusbSmoke", "HusbPassSmoke", "HusbAlcohol", "HusbDrugUse"],
    "Psychosocial Factors": [
        "WifeEnvExpose", "WifeStress", "WifeRelatStress", "WifeFinance", "WifeReady",
        "HusbEnvExpose", "HusbStress", "HusbRelatStress", "HusbFinance", "HusbReady"],
}

# ---- Category Value Labels (coded value -> display label) ----
_EDU_LABELS = {
    "A": "Illiterate", "B": "Elementary School", "C": "Junior High School",
    "D": "High School / Vocational", "E": "Bachelor's Degree", "F": "Graduate Degree",
}
_OCC_LABELS = {
    "A": "Farmer", "B": "Worker", "C": "Teacher / Civil Servant / Office",
    "D": "Service Industry", "E": "Business", "F": "Housework",
}
_WIFE_ABO_LABELS = {
    "Type O": "Type O", "Type A": "Type A", "Type B": "Type B", "Type AB": "Type AB",
}
_HUSB_ABO_LABELS = {
    "A": "Type O", "B": "Type A", "C": "Type B", "D": "Type AB",
}
_HEPB_LABELS = {
    "A": "HBeAg Positive", "B": "Minor Hepatitis B", "C": "Past Infection",
    "D": "Recovery / Vaccination", "E": "Susceptible Population",
}

VALUE_LABELS = {
    "WifeEdu": _EDU_LABELS, "HusbandEdu": _EDU_LABELS,
    "WifeOcc": _OCC_LABELS, "HusbandOcc": _OCC_LABELS,
    "WifeABO": _WIFE_ABO_LABELS, "HusbABO": _HUSB_ABO_LABELS,
    "WifeHepB": _HEPB_LABELS, "HusbHepB": _HEPB_LABELS,
}

def cat_label(var, raw_value):
    """Translate a coded category value to its display label."""
    raw_str = str(raw_value).strip()
    if var in VALUE_LABELS and raw_str in VALUE_LABELS[var]:
        return VALUE_LABELS[var][raw_str]
    return raw_str

def cat_sort_key(var, cat_str):
    """Sort key: use original coded order if value labels exist."""
    if var in VALUE_LABELS:
        order = list(VALUE_LABELS[var].keys())
        raw = str(cat_str).strip()
        for code, label in VALUE_LABELS[var].items():
            if label == raw:
                try: return order.index(code)
                except ValueError: pass
        try: return order.index(raw)
        except ValueError: pass
    return str(cat_str)

ALL_ANALYSIS_VARS = CONTINUOUS_VARS + ORDINAL_VARS + NOMINAL_VARS + BINARY_VARS

# Dataset display names
DS_TRAINING = "Training"
DS_INTERNAL = "Internal Validation"
DS_TEMPORAL = "Temporal Validation"


# ============================================================
# Helper Functions
# ============================================================
def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def compute_smd(x1, x2):
    m1, m2 = np.nanmean(x1), np.nanmean(x2)
    s1, s2 = np.nanstd(x1, ddof=1), np.nanstd(x2, ddof=1)
    pooled = np.sqrt((s1**2 + s2**2) / 2)
    return abs(m1 - m2) / pooled if pooled > 0 else 0.0


def compute_smd_binary(p1, p2):
    pooled = np.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / 2)
    return abs(p1 - p2) / pooled if pooled > 0 else 0.0


def format_p(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def get_outcome_binary(df):
    if OUTCOME_VAR not in df.columns:
        return pd.Series(dtype=float)
    return df[OUTCOME_VAR].map(
        {True: 1, False: 0, "TRUE": 1, "FALSE": 0, "True": 1, "False": 0, 1: 1, 0: 0}
    ).astype(float)


# ============================================================
# Core Descriptive Statistics Functions
# ============================================================
def describe_continuous(datasets, var):
    results = {}
    all_vals = []
    for name, df in datasets.items():
        if var not in df.columns:
            continue
        vals = safe_numeric(df[var]).dropna()
        all_vals.append(vals)
        n = len(vals)
        miss = len(df) - n
        if n > 0:
            results[name] = {
                "N_obs": n, "Missing": miss, "Missing%": miss / len(df) * 100,
                "Mean": np.mean(vals), "SD": np.std(vals, ddof=1),
                "Median": np.median(vals),
                "Q1": np.percentile(vals, 25), "Q3": np.percentile(vals, 75),
                "Min": np.min(vals), "Max": np.max(vals),
                "Mean_SD": f"{np.mean(vals):.2f} \u00b1 {np.std(vals, ddof=1):.2f}",
                "Median_IQR": f"{np.median(vals):.2f} ({np.percentile(vals, 25):.2f}-{np.percentile(vals, 75):.2f})",
            }
        else:
            results[name] = {
                "N_obs": 0, "Missing": miss, "Missing%": miss / len(df) * 100,
                "Mean": np.nan, "SD": np.nan, "Median": np.nan,
                "Q1": np.nan, "Q3": np.nan, "Min": np.nan, "Max": np.nan,
                "Mean_SD": "—", "Median_IQR": "—",
            }

    p_kw = np.nan
    valid = [v for v in all_vals if len(v) > 0]
    if len(valid) >= 2:
        try:
            _, p_kw = stats.kruskal(*valid)
        except Exception:
            pass

    names = list(datasets.keys())
    smd_int, smd_temp = np.nan, np.nan
    if len(names) >= 2:
        v0 = safe_numeric(datasets[names[0]][var]).dropna() if var in datasets[names[0]].columns else pd.Series()
        v1 = safe_numeric(datasets[names[1]][var]).dropna() if var in datasets[names[1]].columns else pd.Series()
        if len(v0) > 0 and len(v1) > 0:
            smd_int = compute_smd(v0, v1)
    if len(names) >= 3:
        v2 = safe_numeric(datasets[names[2]][var]).dropna() if var in datasets[names[2]].columns else pd.Series()
        if len(v0) > 0 and len(v2) > 0:
            smd_temp = compute_smd(v0, v2)

    return results, p_kw, smd_int, smd_temp


def describe_categorical(datasets, var):
    results = {}
    all_cats = set()
    for name, df in datasets.items():
        if var not in df.columns:
            continue
        s = df[var].copy()
        s = s.map({True: "True", False: "False", 1: "True", 0: "False"}).fillna(s)
        vc = s.value_counts(dropna=True)
        n = vc.sum()
        miss = df[var].isna().sum()
        all_cats.update(vc.index)
        results[name] = {
            "N": n, "Missing": miss,
            "Missing%": miss / len(df) * 100 if len(df) > 0 else 0,
            "counts": vc, "pcts": vc / n * 100 if n > 0 else vc * 0,
        }
    all_cats = sorted(all_cats, key=lambda x: cat_sort_key(var, x))

    p_chi = np.nan
    names = list(datasets.keys())
    if len(names) >= 2 and len(all_cats) >= 2:
        try:
            ctab = pd.DataFrame(index=all_cats)
            for name in names:
                if name in results:
                    ctab[name] = results[name]["counts"].reindex(all_cats, fill_value=0)
            _, p_chi, _, _ = stats.chi2_contingency(ctab.values)
        except Exception:
            pass

    smd_int, smd_temp = np.nan, np.nan
    if len(all_cats) == 2 and len(names) >= 2:
        true_cat = "True" if "True" in all_cats else all_cats[-1]
        p_vals = []
        for name in names:
            if name in results and results[name]["N"] > 0:
                cnt = results[name]["counts"].get(true_cat, 0)
                p_vals.append(cnt / results[name]["N"])
            else:
                p_vals.append(np.nan)
        if len(p_vals) >= 2 and not np.isnan(p_vals[0]) and not np.isnan(p_vals[1]):
            smd_int = compute_smd_binary(p_vals[0], p_vals[1])
        if len(p_vals) >= 3 and not np.isnan(p_vals[0]) and not np.isnan(p_vals[2]):
            smd_temp = compute_smd_binary(p_vals[0], p_vals[2])

    return results, all_cats, p_chi, smd_int, smd_temp


# ============================================================
# Split Raw Data (identical logic to imputation script)
# ============================================================
def split_raw_data(raw_path):
    print(f"\nReading raw data: {raw_path}")
    if raw_path.endswith(".csv"):
        df_raw = pd.read_csv(raw_path)
    else:
        df_raw = pd.read_excel(raw_path)
    print(f"  Raw data dimensions: {df_raw.shape}")

    df_raw = df_raw.dropna(subset=[OUTCOME_VAR])
    print(f"  After excluding missing outcome: {len(df_raw):,}")

    df_raw[DATE_VAR] = pd.to_datetime(df_raw[DATE_VAR], errors="coerce")
    df_raw["Year"] = df_raw[DATE_VAR].dt.year
    df_raw[OUTCOME_VAR] = df_raw[OUTCOME_VAR].map(
        {True: 1, False: 0, "TRUE": 1, "FALSE": 0, 1: 1, 0: 0})

    df_temporal = df_raw[df_raw["Year"] == 2019].copy()
    df_dev = df_raw[df_raw["Year"].between(2014, 2018) | df_raw["Year"].isnull()].copy()

    df_train, df_internal_val = train_test_split(
        df_dev, test_size=1 - TRAIN_RATIO,
        stratify=df_dev[OUTCOME_VAR].astype(int), random_state=RANDOM_STATE)

    for subset in [df_train, df_internal_val, df_temporal]:
        subset.drop(columns=["Year"], inplace=True, errors="ignore")

    raw_datasets = {
        DS_TRAINING: df_train, DS_INTERNAL: df_internal_val, DS_TEMPORAL: df_temporal}

    print("\n  Pre-imputation dataset split:")
    for name, data in raw_datasets.items():
        y = data[OUTCOME_VAR].astype(int)
        print(f"    {name:25s}: N={len(data):>7,}  |  SA={y.sum():>5,} ({y.mean()*100:.2f}%)")
    return raw_datasets


# ============================================================
# Excel Styles
# ============================================================
class Styles:
    header_font = Font(bold=True, size=11, color="FFFFFF", name="Arial")
    header_fill = PatternFill("solid", fgColor="4472C4")
    section_fill = PatternFill("solid", fgColor="E2EFDA")
    section_font = Font(bold=True, size=11, name="Arial", color="375623")
    normal_font = Font(size=10, name="Arial")
    bold_font = Font(size=10, name="Arial", bold=True)
    sig_font = Font(size=10, name="Arial", bold=True, color="C00000")
    good_font = Font(size=10, name="Arial", bold=True, color="006100")
    italic_font = Font(size=10, name="Arial", italic=True, color="808080")
    note_font = Font(size=9, name="Arial", italic=True, color="666666")
    thin_border = Border(
        left=Side(style="thin", color="BFBFBF"), right=Side(style="thin", color="BFBFBF"),
        top=Side(style="thin", color="BFBFBF"), bottom=Side(style="thin", color="BFBFBF"))
    center = Alignment(horizontal="center")
    wrap_center = Alignment(horizontal="center", wrap_text=True)
    left = Alignment(horizontal="left")

S = Styles()


def write_header(ws, row, headers):
    for c, h in enumerate(headers, 1):
        cell = ws.cell(row=row, column=c, value=h)
        cell.font = S.header_font; cell.fill = S.header_fill
        cell.alignment = S.wrap_center; cell.border = S.thin_border


def wcell(ws, row, col, value, font=None, fill=None, align=None):
    cell = ws.cell(row=row, column=col, value=value)
    cell.font = font or S.normal_font; cell.border = S.thin_border
    cell.alignment = align or S.center
    if fill:
        cell.fill = fill
    return cell


# ============================================================
# Sheet Writers
# ============================================================
def write_overview(wb, raw_ds, imp_ds):
    ws = wb.active
    ws.title = "Dataset Overview"
    ds_names = list(raw_ds.keys())

    headers = ["Metric"]
    for n in ds_names:
        headers += [f"{n} (Pre)", f"{n} (Post)"]
    write_header(ws, 1, headers)

    def add_row(ws, row, label, pre, post):
        wcell(ws, row, 1, label, font=S.bold_font, align=S.left)
        col = 2
        for i in range(len(ds_names)):
            wcell(ws, row, col, pre[i] if i < len(pre) else "—")
            wcell(ws, row, col + 1, post[i] if i < len(post) else "—")
            col += 2

    row = 2
    add_row(ws, row, "Sample Size (N)",
            [f"{len(raw_ds[n]):,}" for n in ds_names],
            [f"{len(imp_ds[n]):,}" if n in imp_ds else "—" for n in ds_names])
    row += 1

    pre_sa, post_sa = [], []
    for n in ds_names:
        y = get_outcome_binary(raw_ds[n])
        pre_sa.append(f"{int(y.sum()):,} ({y.mean()*100:.2f}%)")
        if n in imp_ds:
            y2 = get_outcome_binary(imp_ds[n])
            post_sa.append(f"{int(y2.sum()):,} ({y2.mean()*100:.2f}%)")
        else:
            post_sa.append("—")
    add_row(ws, row, "SA Positive n(%)", pre_sa, post_sa); row += 1

    pre_neg, post_neg = [], []
    for n in ds_names:
        y = get_outcome_binary(raw_ds[n])
        pre_neg.append(f"{int(len(raw_ds[n])-y.sum()):,} ({(1-y.mean())*100:.2f}%)")
        if n in imp_ds:
            y2 = get_outcome_binary(imp_ds[n])
            post_neg.append(f"{int(len(imp_ds[n])-y2.sum()):,} ({(1-y2.mean())*100:.2f}%)")
        else:
            post_neg.append("—")
    add_row(ws, row, "SA Negative n(%)", pre_neg, post_neg); row += 1

    add_row(ws, row, "Total Analysis Variables",
            [str(len([c for c in ALL_ANALYSIS_VARS if c in raw_ds[n].columns])) for n in ds_names],
            [str(len([c for c in ALL_ANALYSIS_VARS if c in imp_ds[n].columns])) if n in imp_ds else "—" for n in ds_names])
    row += 1
    add_row(ws, row, "  Continuous Variables",
            [str(len([c for c in CONTINUOUS_VARS if c in raw_ds[n].columns])) for n in ds_names],
            [str(len([c for c in CONTINUOUS_VARS if c in imp_ds[n].columns])) if n in imp_ds else "—" for n in ds_names])
    row += 1
    add_row(ws, row, "  Categorical Variables",
            [str(len([c for c in CATEGORICAL_VARS if c in raw_ds[n].columns])) for n in ds_names],
            [str(len([c for c in CATEGORICAL_VARS if c in imp_ds[n].columns])) if n in imp_ds else "—" for n in ds_names])
    row += 1

    pre_m, post_m = [], []
    for n in ds_names:
        ac = [c for c in ALL_ANALYSIS_VARS if c in raw_ds[n].columns]
        t = len(raw_ds[n]) * len(ac) if ac else 1
        m = raw_ds[n][ac].isna().sum().sum()
        pre_m.append(f"{int(m):,} ({m/t*100:.2f}%)")
        if n in imp_ds:
            ac2 = [c for c in ALL_ANALYSIS_VARS if c in imp_ds[n].columns]
            t2 = len(imp_ds[n]) * len(ac2) if ac2 else 1
            m2 = imp_ds[n][ac2].isna().sum().sum()
            post_m.append(f"{int(m2):,} ({m2/t2*100:.2f}%)")
        else:
            post_m.append("—")
    add_row(ws, row, "Missing Cells n(%)", pre_m, post_m); row += 1

    pre_mv, post_mv = [], []
    for n in ds_names:
        ac = [c for c in ALL_ANALYSIS_VARS if c in raw_ds[n].columns]
        pre_mv.append(str(sum(1 for c in ac if raw_ds[n][c].isna().any())))
        if n in imp_ds:
            ac2 = [c for c in ALL_ANALYSIS_VARS if c in imp_ds[n].columns]
            post_mv.append(str(sum(1 for c in ac2 if imp_ds[n][c].isna().any())))
        else:
            post_mv.append("—")
    add_row(ws, row, "Variables with Missing", pre_mv, post_mv)

    ws.column_dimensions["A"].width = 28
    for i in range(1, len(headers)):
        ws.column_dimensions[get_column_letter(i + 1)].width = 24
    ws.freeze_panes = "B2"


def write_continuous_sheet(wb, datasets, sheet_name):
    ws = wb.create_sheet(sheet_name)
    ds_names = list(datasets.keys())

    headers = ["Variable", "Label"]
    for n in ds_names:
        headers += [f"{n} N(Miss%)", f"{n} Mean\u00b1SD", f"{n} Median(IQR)", f"{n} Min-Max"]
    headers += ["P (KW)", "SMD(Train vs IntVal)", "SMD(Train vs TempVal)"]
    write_header(ws, 1, headers)

    row = 2
    for var in CONTINUOUS_VARS:
        if not any(var in datasets[n].columns for n in ds_names):
            continue
        res, p_kw, smd_int, smd_temp = describe_continuous(datasets, var)
        wcell(ws, row, 1, var, font=S.bold_font, align=S.left)
        wcell(ws, row, 2, VAR_LABELS.get(var, ""), align=S.left)
        col = 3
        for n in ds_names:
            if n in res:
                r = res[n]
                wcell(ws, row, col, f"{r['N_obs']:,} ({r['Missing%']:.1f}%)")
                wcell(ws, row, col + 1, r["Mean_SD"])
                wcell(ws, row, col + 2, r["Median_IQR"])
                wcell(ws, row, col + 3, f"{r['Min']:.2f}-{r['Max']:.2f}")
            else:
                for o in range(4):
                    wcell(ws, row, col + o, "—")
            col += 4
        pc = wcell(ws, row, col, format_p(p_kw))
        if not np.isnan(p_kw) and p_kw < 0.05: pc.font = S.sig_font
        col += 1
        for sv in [smd_int, smd_temp]:
            c = wcell(ws, row, col, f"{sv:.3f}" if not np.isnan(sv) else "")
            if not np.isnan(sv) and sv > 0.1: c.font = S.sig_font
            col += 1
        row += 1

    ws.column_dimensions["A"].width = 18; ws.column_dimensions["B"].width = 28
    for i in range(2, len(headers)):
        ws.column_dimensions[get_column_letter(i + 1)].width = 24
    ws.freeze_panes = "C2"


def write_categorical_sheet(wb, datasets, sheet_name):
    ws = wb.create_sheet(sheet_name)
    ds_names = list(datasets.keys())
    headers = ["Variable", "Category"]
    for n in ds_names:
        headers += [f"{n} n(%)"]
    headers += ["P (Chi2)", "SMD(Train vs IntVal)", "SMD(Train vs TempVal)"]
    write_header(ws, 1, headers)
    p_col = 3 + len(ds_names)
    row = 2

    for var in BINARY_VARS + ORDINAL_VARS + NOMINAL_VARS:
        if not any(var in datasets[n].columns for n in ds_names):
            continue
        res, cats, p_chi, smd_int, smd_temp = describe_categorical(datasets, var)

        wcell(ws, row, 1, var, font=S.section_font, fill=S.section_fill, align=S.left)
        wcell(ws, row, 2, VAR_LABELS.get(var, ""), font=Font(size=10, name="Arial", italic=True), fill=S.section_fill)
        for cc in range(3, p_col):
            wcell(ws, row, cc, "", fill=S.section_fill)
        pc = wcell(ws, row, p_col, format_p(p_chi))
        if not np.isnan(p_chi) and p_chi < 0.05: pc.font = S.sig_font
        s1c = wcell(ws, row, p_col + 1, f"{smd_int:.3f}" if not np.isnan(smd_int) else "")
        if not np.isnan(smd_int) and smd_int > 0.1: s1c.font = S.sig_font
        s2c = wcell(ws, row, p_col + 2, f"{smd_temp:.3f}" if not np.isnan(smd_temp) else "")
        if not np.isnan(smd_temp) and smd_temp > 0.1: s2c.font = S.sig_font
        row += 1

        # Missing row
        wcell(ws, row, 1, ""); wcell(ws, row, 2, "Missing n(%)", font=S.italic_font)
        for di, n in enumerate(ds_names):
            if n in res:
                mn = res[n]["Missing"]; mp = res[n]["Missing%"]
                v = f"{mn:,} ({mp:.1f}%)" if mn > 0 else "0 (0.0%)"
                wcell(ws, row, 3 + di, v, font=S.sig_font if mn > 0 else S.normal_font)
            else:
                wcell(ws, row, 3 + di, "—")
        for cc in range(p_col, p_col + 3): wcell(ws, row, cc, "")
        row += 1

        for cat in cats:
            wcell(ws, row, 1, "")
            wcell(ws, row, 2, cat_label(var, cat), align=Alignment(horizontal="left", indent=2))
            for di, n in enumerate(ds_names):
                if n in res and res[n]["N"] > 0:
                    cnt = res[n]["counts"].get(cat, 0)
                    pct = res[n]["pcts"].get(cat, 0)
                    wcell(ws, row, 3 + di, f"{int(cnt):,} ({pct:.1f}%)")
                else:
                    wcell(ws, row, 3 + di, "—")
            for cc in range(p_col, p_col + 3): wcell(ws, row, cc, "")
            row += 1

    ws.column_dimensions["A"].width = 22; ws.column_dimensions["B"].width = 30
    for i in range(len(ds_names)):
        ws.column_dimensions[get_column_letter(3 + i)].width = 22
    for i in range(3):
        ws.column_dimensions[get_column_letter(p_col + i)].width = 22
    ws.freeze_panes = "C2"


def write_missing_comparison(wb, raw_ds, imp_ds):
    ws = wb.create_sheet("Missing Rate Comparison")
    ds_names = list(raw_ds.keys())
    headers = ["Variable", "Type"]
    for n in ds_names:
        headers += [f"{n} Pre n(%)", f"{n} Post n(%)"]
    write_header(ws, 1, headers)
    row = 2
    for var in ALL_ANALYSIS_VARS:
        if not any(var in raw_ds[n].columns for n in ds_names):
            continue
        vtype = ("Continuous" if var in CONTINUOUS_VARS else
                 "Ordinal" if var in ORDINAL_VARS else
                 "Nominal" if var in NOMINAL_VARS else "Binary")
        wcell(ws, row, 1, var, align=S.left); wcell(ws, row, 2, vtype)
        col = 3
        for n in ds_names:
            for dg in [raw_ds, imp_ds]:
                if n in dg and var in dg[n].columns:
                    mn = dg[n][var].isna().sum(); mp = mn / len(dg[n]) * 100
                    v = f"{mn:,} ({mp:.1f}%)"
                    wcell(ws, row, col, v, font=S.sig_font if mp > 0 else S.normal_font)
                else:
                    wcell(ws, row, col, "—")
                col += 1
        row += 1
    ws.column_dimensions["A"].width = 22; ws.column_dimensions["B"].width = 10
    for i in range(2, len(headers)):
        ws.column_dimensions[get_column_letter(i + 1)].width = 22
    ws.freeze_panes = "C2"


def write_imputation_effect(wb, raw_ds, imp_ds):
    ws = wb.create_sheet("Imputation Effect")
    ds_names = list(raw_ds.keys())
    headers = ["Variable", "Type"]
    for n in ds_names:
        headers += [f"{n} Delta Mean", f"{n} Delta SD", f"{n} Delta Median"]
    write_header(ws, 1, headers)
    row = 2

    for var in CONTINUOUS_VARS:
        if not any(var in raw_ds[n].columns for n in ds_names):
            continue
        wcell(ws, row, 1, var, align=S.left); wcell(ws, row, 2, "Continuous")
        col = 3
        for n in ds_names:
            rv = safe_numeric(raw_ds[n][var]).dropna() if (n in raw_ds and var in raw_ds[n].columns) else pd.Series()
            iv = safe_numeric(imp_ds[n][var]).dropna() if (n in imp_ds and var in imp_ds[n].columns) else pd.Series()
            if len(rv) > 0 and len(iv) > 0:
                wcell(ws, row, col, f"{np.mean(iv)-np.mean(rv):+.3f}")
                wcell(ws, row, col+1, f"{np.std(iv,ddof=1)-np.std(rv,ddof=1):+.3f}")
                wcell(ws, row, col+2, f"{np.median(iv)-np.median(rv):+.3f}")
            else:
                for o in range(3): wcell(ws, row, col+o, "—")
            col += 3
        row += 1

    row += 1
    wcell(ws, row, 1, "Binary Variables (Delta True%, pp)", font=S.section_font, fill=S.section_fill, align=S.left)
    for c in range(2, 3 + len(ds_names)*3): wcell(ws, row, c, "", fill=S.section_fill)
    row += 1

    bh = ["Variable", "Type"]
    for n in ds_names: bh += [f"{n} Delta True%", "", ""]
    write_header(ws, row, bh); row += 1

    for var in BINARY_VARS:
        if not any(var in raw_ds[n].columns for n in ds_names):
            continue
        wcell(ws, row, 1, var, align=S.left); wcell(ws, row, 2, "Binary")
        col = 3
        for n in ds_names:
            rs = raw_ds[n][var] if (n in raw_ds and var in raw_ds[n].columns) else pd.Series()
            ims = imp_ds[n][var] if (n in imp_ds and var in imp_ds[n].columns) else pd.Series()
            rm = rs.map({True:1,False:0,"TRUE":1,"FALSE":0,"True":1,"False":0,1:1,0:0})
            im = ims.map({True:1,False:0,"TRUE":1,"FALSE":0,"True":1,"False":0,1:1,0:0})
            rp = rm.dropna().mean()*100 if rm.dropna().shape[0]>0 else np.nan
            ip = im.dropna().mean()*100 if im.dropna().shape[0]>0 else np.nan
            if not np.isnan(rp) and not np.isnan(ip):
                wcell(ws, row, col, f"{ip-rp:+.2f}pp")
            else:
                wcell(ws, row, col, "—")
            wcell(ws, row, col+1, ""); wcell(ws, row, col+2, "")
            col += 3
        row += 1

    ws.column_dimensions["A"].width = 22; ws.column_dimensions["B"].width = 10
    for i in range(2, 3+len(ds_names)*3):
        ws.column_dimensions[get_column_letter(i+1)].width = 16
    ws.freeze_panes = "C2"


def write_smd_sheet(wb, imp_ds):
    ws = wb.create_sheet("SMD Summary")
    ds_names = list(imp_ds.keys())
    headers = ["Variable", "Type", "SMD (Train vs IntVal)", "SMD (Train vs TempVal)", "Balanced (<0.1)?"]
    write_header(ws, 1, headers)
    row = 2; records = []

    for var in CONTINUOUS_VARS:
        if not any(var in imp_ds[n].columns for n in ds_names): continue
        _, _, si, st = describe_continuous(imp_ds, var)
        records.append((var, "Continuous", si, st))
    for var in BINARY_VARS:
        if not any(var in imp_ds[n].columns for n in ds_names): continue
        _, _, _, si, st = describe_categorical(imp_ds, var)
        records.append((var, "Binary", si, st))

    for var, vt, s1, s2 in records:
        wcell(ws, row, 1, var, align=S.left); wcell(ws, row, 2, vt)
        for ci, sv in [(3,s1),(4,s2)]:
            c = wcell(ws, row, ci, f"{sv:.4f}" if not np.isnan(sv) else "")
            if not np.isnan(sv) and sv > 0.1: c.font = S.sig_font
        bal = "Yes" if (not np.isnan(s1) and s1<=0.1 and not np.isnan(s2) and s2<=0.1) else "No"
        if np.isnan(s1) or np.isnan(s2): bal = "N/A"
        wcell(ws, row, 5, bal, font=S.good_font if bal=="Yes" else S.sig_font)
        row += 1

    row += 1; wcell(ws, row, 1, "Summary", font=S.bold_font, align=S.left); row += 1
    nb = sum(1 for _,_,s1,s2 in records if not np.isnan(s1) and s1<=0.1 and not np.isnan(s2) and s2<=0.1)
    nt = sum(1 for _,_,s1,s2 in records if not np.isnan(s1) and not np.isnan(s2))
    txt = f"SMD <= 0.1: {nb}/{nt} ({nb/nt*100:.1f}%)" if nt > 0 else "N/A"
    wcell(ws, row, 1, txt, align=S.left)

    ws.column_dimensions["A"].width = 22; ws.column_dimensions["B"].width = 12
    for i in range(3,6): ws.column_dimensions[get_column_letter(i)].width = 24
    ws.freeze_panes = "A2"


def write_table1(wb, datasets, sheet_name):
    """Table 1 style: grouped continuous + categorical in one sheet."""
    ws = wb.create_sheet(sheet_name)
    ds_names = list(datasets.keys())
    headers = ["Group", "Variable", "Label"]
    for n in ds_names:
        headers += [f"{n} (N={len(datasets[n]):,})"]
    headers += ["P-value", "Test"]
    write_header(ws, 1, headers)
    p_col = 4 + len(ds_names); test_col = p_col + 1
    row = 2

    for gname, gvars in VAR_GROUPS.items():
        wcell(ws, row, 1, gname, font=S.section_font, fill=S.section_fill, align=S.left)
        for c in range(2, test_col+1): wcell(ws, row, c, "", fill=S.section_fill)
        row += 1

        for var in gvars:
            if not any(var in datasets[n].columns for n in ds_names):
                continue
            label = VAR_LABELS.get(var, "")

            if var in CONTINUOUS_VARS:
                res, p_kw, _, _ = describe_continuous(datasets, var)
                wcell(ws, row, 1, ""); wcell(ws, row, 2, var, font=S.bold_font, align=S.left)
                wcell(ws, row, 3, label, align=S.left)
                col = 4
                for n in ds_names:
                    wcell(ws, row, col, res[n]["Median_IQR"] if n in res else "—")
                    col += 1
                pc = wcell(ws, row, p_col, format_p(p_kw))
                if not np.isnan(p_kw) and p_kw < 0.05: pc.font = S.sig_font
                wcell(ws, row, test_col, "KW"); row += 1

            elif var in BINARY_VARS:
                res, cats, p_chi, _, _ = describe_categorical(datasets, var)
                true_cat = "True" if "True" in cats else (cats[-1] if cats else None)
                wcell(ws, row, 1, ""); wcell(ws, row, 2, var, font=S.bold_font, align=S.left)
                wcell(ws, row, 3, label, align=S.left)
                col = 4
                for n in ds_names:
                    if n in res and res[n]["N"]>0 and true_cat:
                        cnt = res[n]["counts"].get(true_cat, 0)
                        pct = res[n]["pcts"].get(true_cat, 0)
                        wcell(ws, row, col, f"{int(cnt):,} ({pct:.1f}%)")
                    else:
                        wcell(ws, row, col, "—")
                    col += 1
                pc = wcell(ws, row, p_col, format_p(p_chi))
                if not np.isnan(p_chi) and p_chi < 0.05: pc.font = S.sig_font
                wcell(ws, row, test_col, "Chi2"); row += 1

            elif var in ORDINAL_VARS + NOMINAL_VARS:
                res, cats, p_chi, _, _ = describe_categorical(datasets, var)
                wcell(ws, row, 1, ""); wcell(ws, row, 2, var, font=S.bold_font, align=S.left)
                wcell(ws, row, 3, label, align=S.left)
                for c in range(4, 4+len(ds_names)): wcell(ws, row, c, "")
                pc = wcell(ws, row, p_col, format_p(p_chi))
                if not np.isnan(p_chi) and p_chi < 0.05: pc.font = S.sig_font
                wcell(ws, row, test_col, "Chi2"); row += 1
                for cat in cats:
                    wcell(ws, row, 1, ""); wcell(ws, row, 2, "")
                    wcell(ws, row, 3, f"  {cat_label(var, cat)}", align=S.left)
                    col = 4
                    for n in ds_names:
                        if n in res and res[n]["N"]>0:
                            cnt = res[n]["counts"].get(cat, 0)
                            pct = res[n]["pcts"].get(cat, 0)
                            wcell(ws, row, col, f"{int(cnt):,} ({pct:.1f}%)")
                        else:
                            wcell(ws, row, col, "—")
                        col += 1
                    wcell(ws, row, p_col, ""); wcell(ws, row, test_col, "")
                    row += 1

    row += 1
    note = ("Note: Continuous variables presented as Median(IQR), P-value from Kruskal-Wallis test; "
            "Binary variables presented as n(%) for positive cases; "
            "Multi-category variables presented as n(%) per level, P-value from Chi-square test.")
    wcell(ws, row, 1, note, font=S.note_font, align=S.left)
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=test_col)

    ws.column_dimensions["A"].width = 20; ws.column_dimensions["B"].width = 22
    ws.column_dimensions["C"].width = 30
    for i in range(len(ds_names)):
        ws.column_dimensions[get_column_letter(4+i)].width = 26
    ws.column_dimensions[get_column_letter(p_col)].width = 12
    ws.column_dimensions[get_column_letter(test_col)].width = 8
    ws.freeze_panes = "D2"


def write_variable_dictionary(wb, raw_ds):
    ws = wb.create_sheet("Variable Dictionary")
    headers = ["No.", "Variable Name", "Label", "Variable Type", "Group", "Value Range / Description"]
    write_header(ws, 1, headers)

    var_to_group = {}
    for gn, gv in VAR_GROUPS.items():
        for v in gv:
            var_to_group[v] = gn

    ref_df = next(iter(raw_ds.values()), None)
    row = 2; seq = 1

    # Outcome
    wcell(ws, row, 1, "", fill=S.section_fill)
    wcell(ws, row, 2, "-- Outcome Variable --", font=S.section_font, fill=S.section_fill, align=S.left)
    for c in range(3, 7): wcell(ws, row, c, "", fill=S.section_fill)
    row += 1
    wcell(ws, row, 1, seq); wcell(ws, row, 2, OUTCOME_VAR, font=S.bold_font, align=S.left)
    wcell(ws, row, 3, VAR_LABELS.get(OUTCOME_VAR, ""), align=S.left)
    wcell(ws, row, 4, "Binary (Outcome)"); wcell(ws, row, 5, "—")
    wcell(ws, row, 6, "True = Spontaneous Abortion, False = No Abortion", align=S.left)
    seq += 1; row += 1

    type_sections = [
        ("-- Continuous Variables --", CONTINUOUS_VARS, "Continuous"),
        ("-- Ordinal Variables --", ORDINAL_VARS, "Ordinal"),
        ("-- Nominal Variables --", NOMINAL_VARS, "Nominal"),
        ("-- Binary Variables --", BINARY_VARS, "Binary"),
    ]

    for stitle, vlist, vtype in type_sections:
        wcell(ws, row, 1, "", fill=S.section_fill)
        wcell(ws, row, 2, stitle, font=S.section_font, fill=S.section_fill, align=S.left)
        for c in range(3, 7): wcell(ws, row, c, "", fill=S.section_fill)
        row += 1

        for var in vlist:
            wcell(ws, row, 1, seq); wcell(ws, row, 2, var, font=S.bold_font, align=S.left)
            wcell(ws, row, 3, VAR_LABELS.get(var, ""), align=S.left)
            wcell(ws, row, 4, vtype)
            wcell(ws, row, 5, var_to_group.get(var, "Other"), align=S.left)
            rdesc = ""
            if ref_df is not None and var in ref_df.columns:
                if vtype == "Continuous":
                    vals = safe_numeric(ref_df[var]).dropna()
                    if len(vals) > 0:
                        rdesc = f"Range: {vals.min():.1f} ~ {vals.max():.1f}"
                elif vtype == "Binary":
                    rdesc = "True / False"
                else:
                    uniq = ref_df[var].dropna().unique()
                    if var in VALUE_LABELS:
                        labeled = [cat_label(var, x) for x in sorted(uniq, key=lambda x: cat_sort_key(var, x))]
                        rdesc = ", ".join(labeled) if len(labeled) <= 10 else f"{len(uniq)} categories"
                    elif len(uniq) <= 10:
                        rdesc = ", ".join(sorted([str(x) for x in uniq]))
                    else:
                        rdesc = f"{len(uniq)} categories"
            wcell(ws, row, 6, rdesc, align=S.left)
            seq += 1; row += 1

    ws.column_dimensions["A"].width = 6; ws.column_dimensions["B"].width = 24
    ws.column_dimensions["C"].width = 32; ws.column_dimensions["D"].width = 16
    ws.column_dimensions["E"].width = 28; ws.column_dimensions["F"].width = 50
    ws.freeze_panes = "A2"


# ============================================================
# Main Report Generator
# ============================================================
def create_full_report(raw_ds, imp_ds, output_path):
    wb = Workbook()
    print("\nGenerating Excel report...")
    print("  Sheet 1:  Dataset Overview");        write_overview(wb, raw_ds, imp_ds)
    print("  Sheet 2:  Pre-Imputation Table 1");  write_table1(wb, raw_ds, "Pre-Imp Table 1")
    print("  Sheet 3:  Post-Imputation Table 1"); write_table1(wb, imp_ds, "Post-Imp Table 1")
    print("  Sheet 4:  Pre-Imp Continuous");       write_continuous_sheet(wb, raw_ds, "Pre-Imp Continuous")
    print("  Sheet 5:  Post-Imp Continuous");      write_continuous_sheet(wb, imp_ds, "Post-Imp Continuous")
    print("  Sheet 6:  Pre-Imp Categorical");      write_categorical_sheet(wb, raw_ds, "Pre-Imp Categorical")
    print("  Sheet 7:  Post-Imp Categorical");     write_categorical_sheet(wb, imp_ds, "Post-Imp Categorical")
    print("  Sheet 8:  Missing Rate Comparison");  write_missing_comparison(wb, raw_ds, imp_ds)
    print("  Sheet 9:  Imputation Effect");        write_imputation_effect(wb, raw_ds, imp_ds)
    print("  Sheet 10: SMD Summary");              write_smd_sheet(wb, imp_ds)
    print("  Sheet 11: Variable Dictionary");      write_variable_dictionary(wb, raw_ds)
    wb.save(output_path)
    print(f"\nReport saved: {output_path}")


def print_console_summary(raw_ds, imp_ds):
    print("\n" + "=" * 100)
    print("DESCRIPTIVE STATISTICS SUMMARY (Pre- vs Post-Imputation)")
    print("=" * 100)
    ds_names = list(raw_ds.keys())
    print(f"\n{'Dataset':<28} {'N':>8}  {'SA%(Pre)':>10}  {'SA%(Post)':>10}  {'Miss%(Pre)':>11}  {'Miss%(Post)':>11}")
    print("-" * 90)
    for n in ds_names:
        N = len(raw_ds[n]); yr = get_outcome_binary(raw_ds[n])
        ac = [c for c in ALL_ANALYSIS_VARS if c in raw_ds[n].columns]
        tr = len(raw_ds[n])*len(ac) if ac else 1
        mr = raw_ds[n][ac].isna().sum().sum()/tr*100
        if n in imp_ds:
            yi = get_outcome_binary(imp_ds[n])
            ac2 = [c for c in ALL_ANALYSIS_VARS if c in imp_ds[n].columns]
            ti = len(imp_ds[n])*len(ac2) if ac2 else 1
            mi = imp_ds[n][ac2].isna().sum().sum()/ti*100
            sa_i = f"{yi.mean()*100:.2f}%"; mi_s = f"{mi:.2f}%"
        else:
            sa_i = "—"; mi_s = "—"
        print(f"  {n:<26} {N:>8,}  {yr.mean()*100:>9.2f}%  {sa_i:>10}  {mr:>10.2f}%  {mi_s:>11}")

    print(f"\nContinuous variable examples (Pre -> Post Mean\u00b1SD):")
    count = 0
    for var in CONTINUOUS_VARS:
        if count >= 5: break
        if not any(var in raw_ds[n].columns for n in ds_names): continue
        rr, _, _, _ = describe_continuous(raw_ds, var)
        ri, _, _, _ = describe_continuous(imp_ds, var)
        print(f"  {var} ({VAR_LABELS.get(var, '')}):")
        for n in ds_names:
            rs = rr[n]["Mean_SD"] if n in rr else "—"
            iss = ri[n]["Mean_SD"] if n in ri else "—"
            print(f"    {n}: {rs}  ->  {iss}")
        count += 1
    print(f"\n  ... (see Excel report for full results)")
    print("=" * 100)


# ============================================================
# Entry Point
# ============================================================
if __name__ == "__main__":

    # ---- File Path Configuration ----
    RAW_DATA_PATH = "./Data/dataset1008.csv"
    TRAIN_IMP_PATH = "./Data/data_training_imputed.csv"
    INTVAL_IMP_PATH = "./Data/data_internal_validation_imputed.csv"
    TEMPVAL_IMP_PATH = "./Data/data_temporal_validation_imputed.csv"
    OUTPUT_PATH = "./Data/statistical_description.xlsx"

    # ---- Step 1: Split raw data into pre-imputation subsets ----
    if os.path.exists(RAW_DATA_PATH):
        raw_datasets = split_raw_data(RAW_DATA_PATH)
    else:
        print(f"\nWarning: Raw data file not found: {RAW_DATA_PATH}")
        print("  Will only describe post-imputation data.")
        raw_datasets = {}

    # ---- Step 2: Read post-imputation data ----
    print("\nReading post-imputation data...")
    imp_datasets = {}
    for path, name in [(TRAIN_IMP_PATH, DS_TRAINING),
                        (INTVAL_IMP_PATH, DS_INTERNAL),
                        (TEMPVAL_IMP_PATH, DS_TEMPORAL)]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            imp_datasets[name] = df
            print(f"  {name}: {path} ({len(df):,} rows)")
        else:
            print(f"  Warning: File not found: {path}")

    if not imp_datasets:
        ALL_PATH = "data_all_imputed_with_splits.csv"
        if os.path.exists(ALL_PATH):
            print(f"\nReading from combined file: {ALL_PATH}")
            df_all = pd.read_csv(ALL_PATH)
            for label, tag in [(DS_TRAINING, "Training"),
                               (DS_INTERNAL, "Internal_Validation"),
                               (DS_TEMPORAL, "Temporal_Validation")]:
                sub = df_all[df_all["Dataset"] == tag]
                if len(sub) > 0:
                    imp_datasets[label] = sub.drop(columns=["Dataset"], errors="ignore")
                    print(f"  {label}: {len(sub):,} rows")

    # ---- Fallback ----
    if not imp_datasets and not raw_datasets:
        print("\nError: No data files found! Check path configuration.")
        sys.exit(1)
    if not imp_datasets:
        print("\nWarning: Post-imputation data not found. Describing pre-imputation only.")
        imp_datasets = raw_datasets
    if not raw_datasets:
        print("\nWarning: Raw data not found. Describing post-imputation only.")
        raw_datasets = imp_datasets

    # ---- Step 3: Output ----
    print_console_summary(raw_datasets, imp_datasets)
    os.makedirs(os.path.dirname(OUTPUT_PATH) if os.path.dirname(OUTPUT_PATH) else ".", exist_ok=True)
    create_full_report(raw_datasets, imp_datasets, OUTPUT_PATH)

    print(f"\nDone!")
    print(f"  Excel report: {OUTPUT_PATH}")
    print(f"  11 Sheets: Overview / Pre-Imp Table1 / Post-Imp Table1 /")
    print(f"    Pre-Imp Continuous / Post-Imp Continuous / Pre-Imp Categorical / Post-Imp Categorical /")
    print(f"    Missing Rate Comparison / Imputation Effect / SMD Summary / Variable Dictionary")


Reading raw data: ./Data/dataset1008.csv
  Raw data dimensions: (402226, 137)
  After excluding missing outcome: 402,226

  Pre-imputation dataset split:
    Training                 : N=360,363  |  SA=11,753 (3.26%)
    Internal Validation      : N= 40,041  |  SA=1,306 (3.26%)
    Temporal Validation      : N=  1,822  |  SA=  262 (14.38%)

Reading post-imputation data...
  Training: ./Data/data_training_imputed.csv (360,363 rows)
  Internal Validation: ./Data/data_internal_validation_imputed.csv (40,041 rows)
  Temporal Validation: ./Data/data_temporal_validation_imputed.csv (1,822 rows)

DESCRIPTIVE STATISTICS SUMMARY (Pre- vs Post-Imputation)

Dataset                             N    SA%(Pre)   SA%(Post)   Miss%(Pre)  Miss%(Post)
------------------------------------------------------------------------------------------
  Training                    360,363       3.26%       3.26%        4.19%        0.00%
  Internal Validation          40,041       3.26%       3.26%        4.19%   